In [ ]:
!pip install pandas selfies torch transformers deepchem rdkit swifter tqdm
!pip install faiss-cpu sentence-transformers pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 58.7 MB/s eta 0:00:00
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=6f12efda6407e74a372c657be46d9106066bafa74e8f63140698ad9dea6a465b
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 97.9 MB/s eta 0:00:00


In [ ]:
!pip install rank_bm25 openai

## 文献专利文本RAG系统 V1

In [ ]:
"""
多索引RAG系统 - 彻底解决内存问题
策略：构建多个独立的小索引，避免单个索引过大
"""

import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict
import re
import os
import gc

import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ============ 配置 ============
GDRIVE_BASE = "/content/drive/MyDrive"
PROJECT_NAME = "PatentRAG"
GDRIVE_PROJECT = f"{GDRIVE_BASE}/{PROJECT_NAME}"
os.makedirs(GDRIVE_PROJECT, exist_ok=True)

print(f"✓ 项目目录: {GDRIVE_PROJECT}")


# ============ 快速构建单个小索引 ============

def build_small_index(index_id: int, docs_per_index: int = 90):
    """
    构建单个小索引（安全！）

    Args:
        index_id: 索引编号 (0-9)
        docs_per_index: 每个索引包含的文档数
    """

    print(f"\n{'='*80}")
    print(f"🔨 构建索引 {index_id}")
    print(f"{'='*80}")

    # 1. 加载模型
    print("🔄 加载模型...")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    dimension = 384

    # 2. 查找文档
    base_path = Path("/content/drive/MyDrive/MinerU")
    all_docs = sorted([f.parent for f in base_path.rglob("full.md")])

    start_idx = index_id * docs_per_index
    end_idx = min(start_idx + docs_per_index, len(all_docs))

    if start_idx >= len(all_docs):
        print(f"⚠️  索引 {index_id} 超出范围")
        return

    my_docs = all_docs[start_idx:end_idx]
    print(f"📝 处理文档 {start_idx+1}-{end_idx} ({len(my_docs)}个)")

    # 3. 创建索引
    index = faiss.IndexFlatIP(dimension)
    documents = []

    # 4. 处理文档
    for doc_dir in tqdm(my_docs, desc=f"索引{index_id}"):
        try:
            # 读取full.md
            full_md = (doc_dir / "full.md").read_text(encoding='utf-8')

            # 简单分块（按段落）
            chunks = [p.strip() for p in full_md.split('\n\n') if len(p.strip()) > 50]

            if not chunks:
                continue

            # 向量化（小批次）
            for i in range(0, len(chunks), 8):
                batch = chunks[i:i+8]
                embeddings = encoder.encode(batch, show_progress_bar=False, convert_to_numpy=True)
                embeddings = embeddings.astype('float32')
                faiss.normalize_L2(embeddings)

                index.add(embeddings)

                # 保存元数据
                for chunk in batch:
                    documents.append({
                        'content': chunk,
                        'metadata': {'doc_id': doc_dir.name, 'title': ''}
                    })
        except Exception as e:
            print(f"\n✗ {doc_dir.name[:30]}: {str(e)[:50]}")

    # 5. 保存这个小索引
    save_path = Path(f"{GDRIVE_PROJECT}/index_{index_id}")
    save_path.mkdir(parents=True, exist_ok=True)

    faiss.write_index(index, str(save_path / "faiss.index"))

    with open(save_path / "documents.pkl", 'wb') as f:
        pickle.dump({'documents': documents, 'dimension': dimension}, f)

    print(f"\n✅ 索引 {index_id} 完成: {len(documents)} 个文档块")
    print(f"💾 保存位置: {save_path}")

    # 清理内存
    del encoder, index, documents
    gc.collect()


# ============ 查询多个索引 ============

def search_all_indices(query: str, top_k: int = 5):
    """搜索所有小索引并合并结果"""

    print(f"🔍 搜索: {query}")

    # 加载模型
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    query_embedding = encoder.encode(query, convert_to_numpy=True)
    query_embedding = query_embedding.astype('float32').reshape(1, -1)
    faiss.normalize_L2(query_embedding)

    all_results = []

    # 搜索每个索引
    for i in range(10):  # 0-9共10个索引
        index_path = Path(f"{GDRIVE_PROJECT}/index_{i}")

        if not index_path.exists():
            continue

        # 加载索引
        index = faiss.read_index(str(index_path / "faiss.index"))
        with open(index_path / "documents.pkl", 'rb') as f:
            data = pickle.load(f)

        # 搜索
        distances, indices = index.search(query_embedding, top_k)

        for dist, idx in zip(distances[0], indices[0]):
            if idx < len(data['documents']):
                all_results.append({
                    'content': data['documents'][idx]['content'],
                    'metadata': data['documents'][idx]['metadata'],
                    'score': float(dist)
                })

    # 按分数排序
    all_results.sort(key=lambda x: x['score'], reverse=True)

    # 返回Top-K
    return all_results[:top_k]


# ============ 使用示例 ============

# 步骤1: 构建所有小索引（一次一个，避免内存问题）
for i in range(10):
    build_small_index(i, docs_per_index=90)

# 步骤2: 查询
results = search_all_indices("Which photoinitiators work better under LED?", top_k=5)

for i, r in enumerate(results, 1):
    print(f"\n[{i}] 分数: {r['score']:.3f}")
    print(f"内容: {r['content'][:200]}...")

✓ 项目目录: /content/drive/MyDrive/PatentRAG

🔨 构建索引 0
🔄 加载模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


✅ 索引 7 完成: 9026 个文档块
💾 保存位置: /content/drive/MyDrive/PatentRAG/index_7

🔨 构建索引 8
🔄 加载模型...
📝 处理文档 721-810 (90个)


索引8: 100%|██████████| 90/90 [12:29<00:00,  8.33s/it]



✅ 索引 8 完成: 6103 个文档块
💾 保存位置: /content/drive/MyDrive/PatentRAG/index_8

🔨 构建索引 9
🔄 加载模型...
📝 处理文档 811-900 (90个)


索引9: 100%|██████████| 90/90 [19:28<00:00, 12.98s/it]



✅ 索引 9 完成: 9784 个文档块
💾 保存位置: /content/drive/MyDrive/PatentRAG/index_9
🔍 搜索: Which photoinitiators work better under LED?

[1] 分数: 0.750
内容: # Research Progress of Photolytic and LED Sensitive Photoinitiators...

[2] 分数: 0.648
内容: Abstract: At present, most of the photoinitiators used in LED photopolymerization systems were multicomponent photoinitiators, and only a few works involved single component photolytic photoinitiators...

[3] 分数: 0.639
内容: [12] C. Dietlin, S. Schweizer, P. Xiao, J. Zhang, F. Morlet- Savary, B. Graff, J. P. Fouassier, J. Lalevee, Photopolymerization upon LEDs: new photoinitiating systems and strategies, Polymer Chemistry...

[4] 分数: 0.638
内容: [0056] Suitable photoinitiators which can be used in combination with the thioxanthone photoinitiator of the present invention include, but are not limited to, those which are capable of absorbing UV ...

[5] 分数: 0.618
内容: Type I photoinitiating abilities of the studied compounds (0.5% w) were examined using RT- FTIR in t

In [ ]:
"""
多索引高级RAG系统
功能：多索引 + BM25混合检索 + 重排序 + DeepSeek
"""

import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict
import re

import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from openai import OpenAI

# ============ 配置 ============
GDRIVE_PROJECT = "/content/drive/MyDrive/PatentRAG"
NUM_INDICES = 10  # 索引数量


class MultiIndexRAG:
    """多索引RAG系统（支持高级功能）"""

    def __init__(self, use_reranker: bool = True, deepseek_api_key: str = None):
        """
        Args:
            use_reranker: 是否使用重排序
            deepseek_api_key: DeepSeek API密钥（可选）
        """
        print("🔄 初始化高级RAG系统...")

        # 向量检索模型
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

        # 重排序模型（可选）
        self.reranker = None
        if use_reranker:
            print("🔄 加载重排序模型...")
            self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
            print("✓ 重排序模型加载完成")

        # DeepSeek LLM（可选）
        self.llm = None
        if deepseek_api_key:
            self.llm = OpenAI(
                api_key=deepseek_api_key,
                base_url="https://api.deepseek.com"
            )
            print("✓ DeepSeek LLM已初始化")

        # BM25索引（延迟加载）
        self.bm25 = None
        self.bm25_docs = None

        print("✅ 系统初始化完成")

    def search_vector(self, query: str, top_k: int = 20) -> List[Dict]:
        """向量检索（多索引）"""

        # 向量化查询
        query_embedding = self.encoder.encode(query, convert_to_numpy=True)
        query_embedding = query_embedding.astype('float32').reshape(1, -1)
        faiss.normalize_L2(query_embedding)

        all_results = []

        # 搜索所有索引
        for i in range(NUM_INDICES):
            index_path = Path(f"{GDRIVE_PROJECT}/index_{i}")

            if not index_path.exists():
                continue

            # 加载索引
            index = faiss.read_index(str(index_path / "faiss.index"))
            with open(index_path / "documents.pkl", 'rb') as f:
                data = pickle.load(f)

            # 搜索
            distances, indices = index.search(query_embedding, top_k // NUM_INDICES + 5)

            for dist, idx in zip(distances[0], indices[0]):
                if idx < len(data['documents']):
                    all_results.append({
                        'content': data['documents'][idx]['content'],
                        'metadata': data['documents'][idx]['metadata'],
                        'score': float(dist),
                        'source': 'vector'
                    })

        # 按分数排序
        all_results.sort(key=lambda x: x['score'], reverse=True)

        return all_results[:top_k]

    def search_bm25(self, query: str, top_k: int = 20) -> List[Dict]:
        """BM25检索"""

        # 延迟构建BM25索引
        if self.bm25 is None:
            print("🔨 构建BM25索引（首次查询需要）...")
            self._build_bm25_index()

        # 分词
        query_tokens = re.findall(r'\b\w+\b', query.lower())

        # BM25评分
        scores = self.bm25.get_scores(query_tokens)

        # 获取Top-K
        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            if scores[idx] > 0:
                results.append({
                    'content': self.bm25_docs[idx]['content'],
                    'metadata': self.bm25_docs[idx]['metadata'],
                    'score': float(scores[idx]),
                    'source': 'bm25'
                })

        return results

    def _build_bm25_index(self):
        """构建BM25索引（从所有小索引中加载文档）"""

        all_docs = []
        tokenized_corpus = []

        for i in range(NUM_INDICES):
            index_path = Path(f"{GDRIVE_PROJECT}/index_{i}")
            if not index_path.exists():
                continue

            with open(index_path / "documents.pkl", 'rb') as f:
                data = pickle.load(f)

            for doc in data['documents']:
                all_docs.append(doc)
                tokens = re.findall(r'\b\w+\b', doc['content'].lower())
                tokenized_corpus.append(tokens)

        self.bm25_docs = all_docs
        self.bm25 = BM25Okapi(tokenized_corpus)

        print(f"✓ BM25索引完成: {len(all_docs):,} 个文档")

    def search_hybrid(
        self,
        query: str,
        top_k: int = 10,
        vector_weight: float = 0.7,
        use_rerank: bool = True
    ) -> List[Dict]:
        """
        混合检索（向量 + BM25 + 重排序）

        Args:
            query: 查询文本
            top_k: 返回结果数
            vector_weight: 向量检索权重（0-1）
            use_rerank: 是否使用重排序
        """

        # 1. 向量检索
        vector_results = self.search_vector(query, top_k=top_k * 2)

        # 2. BM25检索
        bm25_results = self.search_bm25(query, top_k=top_k * 2)

        # 3. 融合（归一化分数）
        def normalize_scores(results):
            if not results:
                return []
            scores = [r['score'] for r in results]
            min_s, max_s = min(scores), max(scores)
            range_s = max_s - min_s if max_s > min_s else 1.0

            for r in results:
                r['score'] = (r['score'] - min_s) / range_s
            return results

        vector_results = normalize_scores(vector_results)
        bm25_results = normalize_scores(bm25_results)

        # 合并（加权）
        merged = {}
        bm25_weight = 1.0 - vector_weight

        for r in vector_results:
            key = r['content'][:100]
            merged[key] = {
                'content': r['content'],
                'metadata': r['metadata'],
                'score': r['score'] * vector_weight,
                'source': 'hybrid'
            }

        for r in bm25_results:
            key = r['content'][:100]
            if key in merged:
                merged[key]['score'] += r['score'] * bm25_weight
            else:
                merged[key] = {
                    'content': r['content'],
                    'metadata': r['metadata'],
                    'score': r['score'] * bm25_weight,
                    'source': 'hybrid'
                }

        results = sorted(merged.values(), key=lambda x: x['score'], reverse=True)[:top_k * 2]

        # 4. 重排序（可选）
        if use_rerank and self.reranker is not None:
            pairs = [[query, r['content'][:500]] for r in results]
            rerank_scores = self.reranker.predict(pairs)

            for r, score in zip(results, rerank_scores):
                r['score'] = float(score)
                r['source'] = 'hybrid_reranked'

            results.sort(key=lambda x: x['score'], reverse=True)

        return results[:top_k]

    def query_with_llm(
        self,
        query: str,
        top_k: int = 5,
        use_hybrid: bool = True
    ) -> tuple:
        """
        使用LLM生成答案

        Returns:
            (answer, retrieved_results)
        """

        if self.llm is None:
            raise ValueError("请先设置 deepseek_api_key")

        # 检索
        if use_hybrid:
            results = self.search_hybrid(query, top_k=top_k)
        else:
            results = self.search_vector(query, top_k=top_k)

        # 构建上下文
        context = "\n\n".join([
            f"[文献 {i+1}] 相关度: {r['score']:.3f}\n"
            f"内容: {r['content'][:800]}"
            for i, r in enumerate(results[:5])
        ])

        # 构建Prompt
        prompt = f"""你是一位专业的化学文献分析助手。请基于以下文献片段回答用户的问题。

要求：
1. 直接回答问题，引用具体文献内容
2. 如果文献中没有直接答案，说明相关信息
3. 保持客观和专业

文献片段：
{context}

用户问题：{query}

回答："""

        # 调用DeepSeek
        response = self.llm.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "你是一位专业的化学文献分析助手。"},
                {"role": "user", "content": prompt}
            ],
            max_tokens=2048,
            temperature=0.7
        )

        answer = response.choices[0].message.content

        return answer, results


# ============ 使用示例 ============

def example_usage():
    """完整使用示例"""

    print("="*80)
    print("🚀 多索引高级RAG系统")
    print("="*80)

    # 1. 初始化（不使用DeepSeek）
    rag = MultiIndexRAG(
        use_reranker=True,
        deepseek_api_key=None  # 暂不使用LLM
    )

    # 2. 测试不同检索策略
    query = "Which photoinitiators work better under LED light sources?"

    print(f"\n{'='*80}")
    print(f"查询: {query}")
    print(f"{'='*80}")

    # 2a. 纯向量检索
    print("\n【策略1】纯向量检索:")
    results = rag.search_vector(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"[{i}] {r['score']:.3f} | {r['content'][:100]}...")

    # 2b. 混合检索 + 重排序
    print("\n【策略2】混合检索 + 重排序:")
    results = rag.search_hybrid(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"[{i}] {r['score']:.3f} | {r['source']} | {r['content'][:100]}...")

    print("\n✅ 测试完成！")


def query_with_deepseek():
    """使用DeepSeek生成答案"""

    # 初始化（带DeepSeek）
    rag = MultiIndexRAG(
        use_reranker=True,
        deepseek_api_key="sk-xxxxxxxxxxxxx"  # 替换为你的API密钥
    )

    query = "Which photoinitiators work better under LED light sources?"

    # 生成答案
    answer, results = rag.query_with_llm(query, top_k=5, use_hybrid=True)

    print(f"\n💡 DeepSeek回答:")
    print("="*80)
    print(answer)
    print("="*80)

    print(f"\n📚 参考文献:")
    for i, r in enumerate(results, 1):
        print(f"[{i}] {r['score']:.3f} | {r['metadata']['doc_id'][:50]}")


# ============ 运行 ============
if __name__ == "__main__":
    # 先安装依赖
    # !pip install rank-bm25 openai -q

    # 运行示例
    # example_usage()
    query_with_deepseek()

# 📝 完整使用流程（在Colab中）

# # ============================================
# # Cell 1: 安装额外依赖
# # ============================================
# !pip install rank-bm25 openai -q

# # ============================================
# # Cell 2: 运行高级RAG（复制上面完整代码）
# # ============================================
# # （粘贴上面的完整代码）

# # ============================================
# # Cell 3: 测试（不使用DeepSeek）
# # ============================================
  # rag = MultiIndexRAG(use_reranker=True, deepseek_api_key=None)

  # # 混合检索测试
  # results = rag.search_hybrid(
  #     "Which photoinitiators work better under LED?",
  #     top_k=5
  # )

  # for i, r in enumerate(results, 1):
  #     print(f"\n[{i}] 分数: {r['score']:.3f} | 来源: {r['source']}")
  #     print(f"内容: {r['content'][:200]}...")

# # ============================================
# # Cell 4: 使用DeepSeek（可选）
# # ============================================
  # rag = MultiIndexRAG(
  #     use_reranker=True,
  #     deepseek_api_key="sk-xxxxxxxxxx"  # 填入你的API密钥
  # )

  # answer, refs = rag.query_with_llm(
  #     "如何选择适合LED光源的光引发剂？",
  #     top_k=5
  # )

  # print("💡 回答:")
  # print(answer)

🔄 初始化高级RAG系统...
🔄 加载重排序模型...
✓ 重排序模型加载完成
✓ DeepSeek LLM已初始化
✅ 系统初始化完成
🔨 构建BM25索引（首次查询需要）...
✓ BM25索引完成: 119,964 个文档

💡 DeepSeek回答:
根据文献内容，以下光引发剂在LED光源下表现更好：

1. **鎓盐类（onium salts）和肟酯类（oxime esters）光引发剂**（文献1）
   - 因其优异的光化学性能成为研究热点
   - 适用于LED光聚合体系

2. **三苯胺-六芳基双咪唑衍生物**（文献2）
   - 可作为氢受体光引发剂
   - 适用于紫外光和LED光源下的自由基光聚合

3. **酰基膦氧化物类**（文献3）
   - 包括但不限于：
     - 2,4,6-三甲基苯甲酰基-二苯基氧化膦
     - 2,4,6-三甲基苯甲酰基-二苯基膦酸盐
     - 双(2,4,6-三甲基苯甲酰基)苯基氧化膦
   - 能够吸收LED光源发射的特定波长（365-405nm）

4. **CMTX（特定硫杂蒽酮类）**（文献4）
   - 实验证明在LED固化下表现良好
   - 相比之下，水溶性光引发剂Irgacure 2959在LED下完全无法固化

需要注意的是，传统工业中常用的可裂解型光引发剂大多在LED发射下光吸收效率较差，需要红移吸收波长来适应LED光源（文献1）。目前开发对LED敏感的新型光引发剂是该领域的迫切需求。

📚 参考文献:
[1] 6.801 | 光可裂解型LED敏感的光引发剂的研究进展_谢洁.pdf-f55e60c8-69ea-4263-9ef
[2] 4.564 | A review of oxime-ester V6_HAL.pdf-81552d8d-4c22-4
[3] 4.476 | WO2015183719A1.pdf-1875eb29-e833-401a-b735-5e92b28
[4] 3.557 | WO2015183719A1.pdf-1875eb29-e833-401a-b735-5e92b28
[5] 2.460 | 光可裂解型LED敏感的光引发剂的研究进展_谢洁.pdf-f55e60c8-69ea-4263-9ef


## 文献专利多模态RAG系统 V2

In [ ]:
# -*- coding: utf-8 -*-
"""
多模态RAG系统 - 基于CLIP
支持：文本检索 + 图片检索 + 图文混合检索
"""

import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
import re
import os
import gc
from PIL import Image

import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

# ============ 配置 ============
GDRIVE_BASE = "/content/drive/MyDrive"
PROJECT_NAME = "MultimodalRAG"
GDRIVE_PROJECT = f"{GDRIVE_BASE}/{PROJECT_NAME}"
os.makedirs(GDRIVE_PROJECT, exist_ok=True)

print(f"✓ 项目目录: {GDRIVE_PROJECT}")


class MultimodalIndexBuilder:
    """多模态索引构建器（文本 + 图片）"""

    def __init__(self, clip_model_name: str = "openai/clip-vit-base-patch32"):
        """
        Args:
            clip_model_name: CLIP模型名称
        """
        print("🔄 加载CLIP模型...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
        self.dimension = 512  # CLIP embedding维度
        print(f"✓ CLIP模型加载完成 (设备: {self.device})")

    def encode_text(self, texts: List[str]) -> np.ndarray:
        """
        用CLIP编码文本

        Args:
            texts: 文本列表

        Returns:
            embeddings: (N, 512) numpy数组
        """
        inputs = self.clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            text_features = self.clip_model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # L2归一化

        return text_features.cpu().numpy()

    def encode_image(self, image_paths: List[str]) -> np.ndarray:
        """
        用CLIP编码图片

        Args:
            image_paths: 图片路径列表

        Returns:
            embeddings: (N, 512) numpy数组
        """
        images = []
        for path in image_paths:
            try:
                img = Image.open(path).convert('RGB')
                images.append(img)
            except Exception as e:
                print(f"✗ 无法加载图片 {path}: {e}")
                # 创建黑色占位图
                images.append(Image.new('RGB', (224, 224)))

        inputs = self.clip_processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            image_features = self.clip_model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        return image_features.cpu().numpy()

    def build_multimodal_index(
        self,
        index_id: int,
        docs_per_index: int = 50,
        base_path: str = "/content/drive/MyDrive/MinerU"
    ):
        """
        构建多模态索引（文本 + 图片统一索引）

        Args:
            index_id: 索引编号
            docs_per_index: 每个索引包含的文档数
            base_path: MinerU输出目录
        """
        print(f"\n{'='*80}")
        print(f"🔨 构建多模态索引 {index_id}")
        print(f"{'='*80}")

        # 查找文档
        base_path = Path(base_path)
        all_docs = sorted([f.parent for f in base_path.rglob("full.md")])

        start_idx = index_id * docs_per_index
        end_idx = min(start_idx + docs_per_index, len(all_docs))

        if start_idx >= len(all_docs):
            print(f"⚠️  索引 {index_id} 超出范围")
            return

        my_docs = all_docs[start_idx:end_idx]
        print(f"📝 处理文档 {start_idx+1}-{end_idx} ({len(my_docs)}个)")

        # 创建索引（文本和图片共用一个索引）
        index = faiss.IndexFlatIP(self.dimension)
        documents = []  # 存储文本块和图片的元数据

        # 处理每个文档
        for doc_dir in tqdm(my_docs, desc=f"索引{index_id}"):
            try:
                # 1. 处理文本
                full_md = (doc_dir / "full.md").read_text(encoding='utf-8')
                text_chunks = [p.strip() for p in full_md.split('\n\n') if len(p.strip()) > 50]

                # 提取图片引用和上下文
                image_contexts = self._extract_image_contexts(full_md, doc_dir)

                # 2. 文本向量化（批量处理）
                if text_chunks:
                    for i in range(0, len(text_chunks), 8):
                        batch = text_chunks[i:i+8]
                        embeddings = self.encode_text(batch)
                        embeddings = embeddings.astype('float32')

                        index.add(embeddings)

                        for chunk in batch:
                            documents.append({
                                'type': 'text',
                                'content': chunk,
                                'metadata': {'doc_id': doc_dir.name, 'title': ''}
                            })

                # 3. 图片向量化
                if image_contexts:
                    image_paths = [ctx['image_path'] for ctx in image_contexts]
                    valid_paths = [p for p in image_paths if Path(p).exists()]

                    if valid_paths:
                        # 批量编码图片
                        for i in range(0, len(valid_paths), 4):  # 图片批量小一些
                            batch_paths = valid_paths[i:i+4]
                            batch_contexts = [ctx for ctx in image_contexts
                                            if ctx['image_path'] in batch_paths]

                            embeddings = self.encode_image(batch_paths)
                            embeddings = embeddings.astype('float32')

                            index.add(embeddings)

                            for ctx in batch_contexts:
                                documents.append({
                                    'type': 'image',
                                    'content': ctx['context'],
                                    'image_path': ctx['image_path'],
                                    'caption': ctx['caption'],
                                    'metadata': {'doc_id': doc_dir.name, 'title': ''}
                                })

            except Exception as e:
                print(f"\n✗ {doc_dir.name[:30]}: {str(e)[:50]}")

        # 保存索引
        save_path = Path(f"{GDRIVE_PROJECT}/index_{index_id}")
        save_path.mkdir(parents=True, exist_ok=True)

        faiss.write_index(index, str(save_path / "multimodal.index"))

        with open(save_path / "documents.pkl", 'wb') as f:
            pickle.dump({'documents': documents, 'dimension': self.dimension}, f)

        text_count = sum(1 for d in documents if d['type'] == 'text')
        image_count = sum(1 for d in documents if d['type'] == 'image')

        print(f"\n✅ 索引 {index_id} 完成:")
        print(f"   - 文本块: {text_count}")
        print(f"   - 图片: {image_count}")
        print(f"   - 总计: {len(documents)}")
        print(f"💾 保存位置: {save_path}")

        # 清理内存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _extract_image_contexts(self, markdown: str, doc_dir: Path) -> List[Dict]:
        """
        从Markdown中提取图片及其上下文

        Args:
            markdown: markdown文本
            doc_dir: 文档目录

        Returns:
            图片上下文列表: [{'image_path': ..., 'caption': ..., 'context': ...}]
        """
        results = []

        # 正则匹配图片: ![caption](images/xxx.png)
        pattern = r'!\[(.*?)\]\((images/[^\)]+)\)'
        matches = re.finditer(pattern, markdown)

        for match in matches:
            caption = match.group(1)
            image_rel_path = match.group(2)
            image_path = doc_dir / image_rel_path

            if not image_path.exists():
                continue

            # 提取图片周围的文本作为上下文（前后各200字符）
            pos = match.start()
            context_start = max(0, pos - 200)
            context_end = min(len(markdown), pos + 200)
            context = markdown[context_start:context_end].strip()

            results.append({
                'image_path': str(image_path),
                'caption': caption,
                'context': context
            })

        return results


class MultimodalRAG:
    """多模态RAG检索系统"""

    def __init__(self, clip_model_name: str = "openai/clip-vit-base-patch32"):
        print("🔄 初始化多模态RAG系统...")

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

        print("✅ 系统初始化完成")

    def search(
        self,
        query: str = None,
        query_image: str = None,
        top_k: int = 10,
        filter_type: str = None  # 'text', 'image', None
    ) -> List[Dict]:
        """
        多模态检索（支持文本查询、图片查询、或混合查询）

        Args:
            query: 文本查询（可选）
            query_image: 图片路径（可选）
            top_k: 返回结果数
            filter_type: 过滤结果类型（'text'只返回文本，'image'只返回图片）

        Returns:
            检索结果列表
        """
        if query is None and query_image is None:
            raise ValueError("必须提供文本查询或图片查询")

        # 1. 编码查询
        if query and query_image:
            # 混合查询：图文各占50%
            text_emb = self._encode_text([query])
            image_emb = self._encode_image([query_image])
            query_embedding = (text_emb + image_emb) / 2
        elif query:
            query_embedding = self._encode_text([query])
        else:
            query_embedding = self._encode_image([query_image])

        query_embedding = query_embedding.astype('float32')

        # 2. 搜索所有索引
        all_results = []

        index_dirs = sorted(Path(GDRIVE_PROJECT).glob("index_*"))
        for index_path in index_dirs:
            if not (index_path / "multimodal.index").exists():
                continue

            # 加载索引
            index = faiss.read_index(str(index_path / "multimodal.index"))
            with open(index_path / "documents.pkl", 'rb') as f:
                data = pickle.load(f)

            # 搜索
            distances, indices = index.search(query_embedding, top_k * 2)

            for dist, idx in zip(distances[0], indices[0]):
                if idx < len(data['documents']):
                    doc = data['documents'][idx]

                    # 类型过滤
                    if filter_type and doc['type'] != filter_type:
                        continue

                    all_results.append({
                        **doc,
                        'score': float(dist)
                    })

        # 3. 排序并返回Top-K
        all_results.sort(key=lambda x: x['score'], reverse=True)
        return all_results[:top_k]

    def _encode_text(self, texts: List[str]) -> np.ndarray:
        """编码文本"""
        inputs = self.clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            features = self.clip_model.get_text_features(**inputs)
            features = features / features.norm(dim=-1, keepdim=True)

        return features.cpu().numpy()

    def _encode_image(self, image_paths: List[str]) -> np.ndarray:
        """编码图片"""
        images = [Image.open(p).convert('RGB') for p in image_paths]
        inputs = self.clip_processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            features = self.clip_model.get_image_features(**inputs)
            features = features / features.norm(dim=-1, keepdim=True)

        return features.cpu().numpy()

    def display_results(self, results: List[Dict]):
        """美化显示检索结果"""
        for i, r in enumerate(results, 1):
            print(f"\n{'='*80}")
            print(f"[{i}] 类型: {r['type'].upper()} | 相关度: {r['score']:.3f}")
            print(f"文档ID: {r['metadata']['doc_id'][:50]}")

            if r['type'] == 'text':
                print(f"内容预览: {r['content'][:200]}...")
            else:
                print(f"图片路径: {r['image_path']}")
                print(f"图片说明: {r['caption']}")
                print(f"上下文: {r['content'][:150]}...")


# ============ 使用示例 ============

def build_all_indices():
    """构建所有多模态索引"""
    builder = MultimodalIndexBuilder()

    # 假设有1000个文档，分成20个索引（每个50个文档）
    for i in range(20):
        builder.build_multimodal_index(i, docs_per_index=50)

        # 清理内存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def example_text_query():
    """示例1: 纯文本查询（可以检索到文本和图片）"""
    rag = MultimodalRAG()

    query = "LED photoinitiators absorption spectrum"
    print(f"\n🔍 文本查询: {query}")

    results = rag.search(query=query, top_k=5)
    rag.display_results(results)


def example_image_query():
    """示例2: 以图搜图/文"""
    rag = MultimodalRAG()

    query_image = "/path/to/query_image.png"
    print(f"\n🖼️  图片查询: {query_image}")

    results = rag.search(query_image=query_image, top_k=5)
    rag.display_results(results)


def example_hybrid_query():
    """示例3: 图文混合查询"""
    rag = MultimodalRAG()

    query = "absorption spectrum"
    query_image = "/path/to/spectrum.png"

    print(f"\n🔍🖼️  混合查询:")
    print(f"   文本: {query}")
    print(f"   图片: {query_image}")

    results = rag.search(query=query, query_image=query_image, top_k=5)
    rag.display_results(results)


def example_filter_results():
    """示例4: 只返回图片或只返回文本"""
    rag = MultimodalRAG()

    query = "chemical structure"

    # 只返回图片
    print(f"\n🖼️  只检索图片:")
    image_results = rag.search(query=query, top_k=5, filter_type='image')
    rag.display_results(image_results)

    # 只返回文本
    print(f"\n📝 只检索文本:")
    text_results = rag.search(query=query, top_k=5, filter_type='text')
    rag.display_results(text_results)


# ============ 运行 ============
if __name__ == "__main__":
    # 安装依赖
    # !pip install transformers torch pillow faiss-cpu -q

    # 步骤1: 构建索引（只需运行一次）
    build_all_indices()

    # 步骤2: 检索测试
    # example_text_query()
    # example_image_query()
    # example_hybrid_query()
    # example_filter_results()

    pass


✓ 项目目录: /content/drive/MyDrive/MultimodalRAG
🔄 加载CLIP模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

索引2: 100%|██████████| 50/50 [10:51<00:00, 13.02s/it]



✅ 索引 2 完成:
   - 文本块: 6301
   - 图片: 790
   - 总计: 7091
💾 保存位置: /content/drive/MyDrive/MultimodalRAG/index_2

🔨 构建多模态索引 3
📝 处理文档 151-200 (50个)


索引3: 100%|██████████| 50/50 [07:49<00:00,  9.39s/it]



✅ 索引 3 完成:
   - 文本块: 4830
   - 图片: 561
   - 总计: 5391
💾 保存位置: /content/drive/MyDrive/MultimodalRAG/index_3

🔨 构建多模态索引 4
📝 处理文档 201-250 (50个)


索引4:  86%|████████▌ | 43/50 [15:15<00:39,  5.61s/it]

## 文献专利 RAG 系统 v3

### v3 相对 v2 的改动
| 模块 | v2 | v3 |
|------|----|---------|
| Sparse存储 | `List[dict]` Python遍历点积 | `scipy.sparse.csr_matrix` 矩阵乘法（快100x）|
| 图片处理 | 仅用caption文本 | **分级策略**：caption够用走文本；机理图/结构图调VLM API生成描述 |
| 评估体系 | 无 | **RAGAS框架**：自动生成测试集 + 四维指标评估 |

In [ ]:
# 强制清除所有相关包，确保没有版本残留
!pip uninstall -y FlagEmbedding transformers -q
!pip install "transformers==4.46.3" -q
!pip install "FlagEmbedding==1.3.5" -q
!pip install faiss-cpu sentence-transformers rank_bm25 jieba openai tqdm scipy -q
!pip install langchain langchain-text-splitters langchain-openai ragas datasets pillow -q

print("✅ 安装完成，现在点击菜单：Runtime → Restart session")
print("⚠️  必须重启！不重启的话import还是读旧版本")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 122.1 MB/s eta 0:00:00
✅ 安装完成，现在点击菜单：Runtime → Restart session
⚠️  必须重启！不重启的话import还是读旧版本


In [ ]:
# ============================================================
# Cell 2: 配置
# ============================================================
import os
from pathlib import Path

GDRIVE_BASE    = "/content/drive/MyDrive"
MINERU_DIR     = f"{GDRIVE_BASE}/MinerU"
PROJECT_DIR    = f"{GDRIVE_BASE}/PatentRAG_v3"
INDEX_DIR      = f"{PROJECT_DIR}/index"
EVAL_DIR       = f"{PROJECT_DIR}/eval"
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,  exist_ok=True)

from google.colab import userdata

# ---- 模型 ----
EMBED_MODEL    = "BAAI/bge-m3"
RERANK_MODEL   = "BAAI/bge-reranker-v2-m3"

# ---- LLM（DeepSeek 用于生成；也用于评估）----
LLM_API_KEY    = userdata.get('deepseek_api_key') # 从 Colab Secrets 获取密钥
LLM_BASE_URL   = "https://api.deepseek.com/v1"
LLM_MODEL      = "deepseek-chat"

# ---- 多模态：用于化学图片描述（调API，无需本地GPU）----
# 建议用 GPT-4o 或 Claude，它们对化学图的理解最好
# 如果不想用，设为 None，图片退回纯caption文本处理
VLM_API_KEY    = userdata.get('kimi_api_key') # 从 Colab Secrets 获取密钥
VLM_PROVIDER   = "kimi"
VLM_BASE_URL = "https://api.moonshot.cn/v1"
VLM_MODEL      = "kimi-k2.5"

# ---- 检索参数 ----
CHUNK_SIZE     = 512
CHUNK_OVERLAP  = 64
TOP_K_RETRIEVE = 30
TOP_K_RERANK   = 5
RRF_K          = 60

# ---- 图片处理策略 ----
# caption长度阈值：超过此值认为caption已足够，不调VLM
CAPTION_SUFFICIENT_LEN = 50

print(f'✅ 配置完成')

✅ 配置完成


In [ ]:
# ============================================================
# Cell 3: 多模态图片处理（Kimi Vision版）
# ============================================================
import base64
import re
from pathlib import Path
from openai import OpenAI  # Kimi 用 OpenAI 兼容接口，直接复用

# Kimi API 配置（OpenAI兼容）


def _init_vlm_client():
    """初始化VLM客户端"""
    if VLM_PROVIDER == "kimi":
        # Kimi 完全兼容 OpenAI SDK，只需改 base_url
        return OpenAI(api_key=VLM_API_KEY, base_url=VLM_BASE_URL)
    return None


_CHEM_IMAGE_KEYWORDS = [
    'mechanism', 'reaction', 'scheme', '机理', '反应路径', '合成路线',
    'structure', '结构式', 'nmr', 'ir', 'uv', 'spectrum', '光谱',
    'cycle', 'pathway', '循环', '路径', 'figure', '图'
]


def _needs_vlm(caption: str, surrounding_text: str) -> bool:
    if VLM_PROVIDER is None:
        return False
    if len(caption) >= CAPTION_SUFFICIENT_LEN:
        return False
    context = (caption + ' ' + surrounding_text).lower()
    return any(kw in context for kw in _CHEM_IMAGE_KEYWORDS)


def describe_image_with_vlm(image_path: str, caption: str, vlm_client) -> str:
    """
    调用 Kimi Vision API 生成化学图片的详细文字描述。
    注意：Kimi Vision 只支持 base64，不支持图片URL。
    """
    try:
        with open(image_path, 'rb') as f:
            img_b64 = base64.b64encode(f.read()).decode()

        ext = Path(image_path).suffix.lower()
        media_type = {
            '.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.gif': 'image/gif',
            '.webp': 'image/webp'
        }.get(ext, 'image/png')

        prompt = (
            f"这是一张来自化学/材料科学文献的图片，原始标注为：'{caption}'。\n"
            "请详细描述图中的内容，重点关注：\n"
            "1. 如果是反应机理图：描述反应步骤、中间体、箭头方向\n"
            "2. 如果是分子结构式：描述官能团、连接方式\n"
            "3. 如果是光谱图：描述峰位、形状特征\n"
            "4. 如果是实验装置/流程图：描述各组成部分\n"
            "请用中文输出，100-200字。"
        )

        # Kimi Vision：base64图片通过 image_url 的 data URI 传入
        resp = vlm_client.chat.completions.create(
            model=VLM_MODEL,  # kimi-k2.5
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            # Kimi 只支持 base64，格式：data:{mime};base64,{data}
                            "url": f"data:{media_type};base64,{img_b64}"
                        }
                    },
                    {"type": "text", "text": prompt}
                ]
            }],
            max_tokens=300,
        )
        return resp.choices[0].message.content.strip()

    except Exception as e:
        print(f'  ⚠️  Kimi Vision 失败 {Path(image_path).name}: {e}，退回caption')
        return caption

In [ ]:
# ============================================================
# Cell 4: 文档解析 & 语义切块（与v2相同，集成VLM图片处理）
# ============================================================
import jieba
from typing import List, Dict
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


def parse_mineru_doc(doc_dir: Path, vlm_client=None) -> List[Dict]:
    """
    解析单个MinerU文档，返回带结构元数据的chunk列表。
    vlm_client: 非None时对化学图片调VLM生成详细描述
    """
    full_md_path = doc_dir / "full.md"
    if not full_md_path.exists():
        return []

    raw_md = full_md_path.read_text(encoding='utf-8')
    doc_id = doc_dir.name
    chunks = []

    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header1"), ("##", "Header2"), ("###", "Header3")
        ],
        strip_headers=False
    )
    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE * 3,
        chunk_overlap=CHUNK_OVERLAP * 3,
        separators=["\n\n", "\n", "。", "；", " ", ""]
    )

    for section in header_splitter.split_text(raw_md):
        headers = section.metadata
        text = section.page_content
        breadcrumb = " > ".join(v for v in headers.values() if v)

        img_pattern = r'!\[(.*?)\]\((images/[^)]+)\)'
        for img_match in re.finditer(img_pattern, text):
            caption = img_match.group(1).strip()
            img_path = str(doc_dir / img_match.group(2))

            # 取图片周围文字作为上下文
            pos = img_match.start()
            surrounding = text[max(0, pos-150):pos+150].strip()

            if not caption:
                caption = surrounding[:100]

            # ---- 分级图片处理 ----
            if vlm_client and _needs_vlm(caption, surrounding) and Path(img_path).exists():
                # 需要VLM：调API获取详细描述
                description = describe_image_with_vlm(img_path, caption, vlm_client)
                img_content = f"[图片|VLM描述] {breadcrumb}\n{description}".strip()
            else:
                # 不需要VLM：使用caption + 周围文本
                img_content = f"[图片] {breadcrumb}\n{caption}\n上下文: {surrounding[:100]}".strip()

            if len(img_content) > 20:
                chunks.append({
                    'content':          img_content,
                    'content_for_bm25': ' '.join(jieba.cut(img_content)),
                    'doc_id':           doc_id,
                    'headers':          headers,
                    'type':             'image',
                    'image_path':       img_path,
                })

        clean_text = re.sub(img_pattern, '', text).strip()
        if len(clean_text) < 30:
            continue

        for sc in char_splitter.split_text(clean_text):
            if len(sc.strip()) < 30:
                continue
            final = f"{breadcrumb}\n{sc}".strip() if breadcrumb else sc.strip()
            chunks.append({
                'content':          final,
                'content_for_bm25': ' '.join(jieba.cut(final)),
                'doc_id':           doc_id,
                'headers':          headers,
                'type':             'text',
                'image_path':       None,
            })

    return chunks


def load_all_documents(mineru_dir: str, use_vlm: bool = True) -> List[Dict]:
    from tqdm import tqdm
    vlm_client = _init_vlm_client() if use_vlm else None
    if vlm_client:
        print(f'🖼️  VLM图片处理已启用（{VLM_PROVIDER}/{VLM_MODEL}），仅对化学图调用')
    else:
        print('🖼️  VLM图片处理未启用，所有图片使用caption文本')

    base = Path(mineru_dir)
    all_doc_dirs = sorted([f.parent for f in base.rglob("full.md")])
    print(f'📂 发现 {len(all_doc_dirs)} 个文档')

    all_chunks, vlm_count = [], 0
    for doc_dir in tqdm(all_doc_dirs, desc='解析文档'):
        try:
            chunks = parse_mineru_doc(doc_dir, vlm_client)
            vlm_count += sum(1 for c in chunks if 'VLM描述' in c['content'])
            all_chunks.extend(chunks)
        except Exception as e:
            print(f'✗ {doc_dir.name}: {e}')

    print(f'✅ 共 {len(all_chunks):,} 个chunks')
    print(f'   其中 VLM描述图片: {vlm_count} 张，普通caption图片: {sum(1 for c in all_chunks if c["type"]=="image") - vlm_count} 张')
    return all_chunks

In [ ]:
# ============================================================
# Cell 5: 构建统一索引（关键升级：Sparse改为CSR矩阵）
# ============================================================
import pickle
import numpy as np
import faiss
import torch
import scipy.sparse as sp
from FlagEmbedding import BGEM3FlagModel
from rank_bm25 import BM25Okapi
from tqdm import tqdm


def build_index(all_chunks: List[Dict], index_dir: str, batch_size: int = 64):
    """
    构建三路统一索引:
    1. FAISS HNSW (BGE-M3 dense, 1024-dim)
    2. scipy.sparse.csr_matrix (BGE-M3 sparse lexical weights)
       - 查询时用矩阵乘法，比List[dict]遍历快 ~100x
    3. BM25Okapi (jieba分词)
    """
    index_dir = Path(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)

    print('🔄 加载 BGE-M3...')
    model = BGEM3FlagModel(
        EMBED_MODEL, use_fp16=True,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

    contents = [c['content'] for c in all_chunks]
    n = len(contents)
    DENSE_DIM = 1024

    # ------ FAISS HNSW ------
    faiss_index = faiss.IndexHNSWFlat(DENSE_DIM, 32)
    faiss_index.hnsw.efSearch = 64
    faiss_index.hnsw.efConstruction = 64

    # ------ 收集Sparse数据（用于构建CSR矩阵）------
    # BGE-M3 的 lexical_weights 是 {token_id(int): weight(float)} 的字典
    # 我们需要知道词表大小才能建矩阵，先收集所有数据再一次性构建
    sparse_rows = []   # 行下标（chunk编号）
    sparse_cols = []   # 列下标（token_id）
    sparse_vals = []   # 权重值
    max_token_id = 0

    print(f'🔄 批量编码 {n:,} 个chunks...')
    for i in tqdm(range(0, n, batch_size), desc='编码进度'):
        batch = contents[i:i+batch_size]
        outputs = model.encode(
            batch, batch_size=batch_size, max_length=512,
            return_dense=True, return_sparse=True, return_colbert_vecs=False
        )

        # Dense → FAISS
        dense = np.array(outputs['dense_vecs'], dtype='float32')
        faiss.normalize_L2(dense)
        faiss_index.add(dense)

        # Sparse → 收集COO格式数据
        for local_j, lex_weights in enumerate(outputs['lexical_weights']):
            global_row = i + local_j
            for token_id, weight in lex_weights.items():
                sparse_rows.append(global_row)
                sparse_cols.append(int(token_id))
                sparse_vals.append(float(weight))
                if int(token_id) > max_token_id:
                    max_token_id = int(token_id)

    # ------ 一次性构建 CSR 矩阵 ------
    # 形状: (n_chunks, vocab_size)
    # 查询时: query_sparse_vec @ sparse_matrix.T → (1, n_chunks) 分数向量
    print('🔨 构建 CSR 稀疏矩阵...')
    vocab_size = max_token_id + 1
    sparse_matrix = sp.csr_matrix(
        (sparse_vals, (sparse_rows, sparse_cols)),
        shape=(n, vocab_size),
        dtype=np.float32
    )
    print(f'   矩阵形状: {sparse_matrix.shape}, 非零元素: {sparse_matrix.nnz:,}')

    # ------ BM25 ------
    print('🔨 构建 BM25...')
    tokenized = [c['content_for_bm25'].split() for c in all_chunks]
    bm25 = BM25Okapi(tokenized)

    # ------ 保存 ------
    print('💾 保存索引...')
    faiss.write_index(faiss_index, str(index_dir / 'hnsw.index'))
    sp.save_npz(str(index_dir / 'sparse.npz'), sparse_matrix)  # scipy原生格式
    with open(index_dir / 'bm25.pkl', 'wb') as f:
        pickle.dump(bm25, f)
    # 保存vocab_size供查询时用
    with open(index_dir / 'vocab_size.txt', 'w') as f:
        f.write(str(vocab_size))

    metadata = [{
        'content': c['content'], 'doc_id': c['doc_id'],
        'headers': c['headers'], 'type': c['type'], 'image_path': c['image_path']
    } for c in all_chunks]
    with open(index_dir / 'metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)

    print(f'\n✅ 索引构建完成！')
    print(f'   FAISS HNSW: {faiss_index.ntotal:,} 条')
    print(f'   Sparse CSR: {sparse_matrix.shape}, nnz={sparse_matrix.nnz:,}')
    print(f'   BM25: {len(tokenized):,} 条')


# ============================================================
# 运行建索引（只需执行一次）
# ============================================================
# all_chunks = load_all_documents(MINERU_DIR, use_vlm=True)
# build_index(all_chunks, INDEX_DIR, batch_size=64)

🖼️  VLM图片处理已启用（openai/gpt-4o），仅对化学图调用
📂 发现 904 个文档


解析文档: 100%|██████████| 904/904 [01:36<00:00,  9.33it/s]


✅ 共 66,259 个chunks
   其中 VLM描述图片: 0 张，普通caption图片: 19044 张
🔄 加载 BGE-M3...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

🔄 批量编码 66,259 个chunks...


流式输出内容被截断，只能显示最后 5000 行内容。
pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 109.87it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 89.91it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 100.70it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 110.21it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 140.08it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 119.29it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 119.77it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 82.30it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 87.67it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 89.49it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 89.57it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 92.44it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 97.83it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 102.10it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 94.78it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 10

🔨 构建 CSR 稀疏矩阵...
   矩阵形状: (66259, 249984), 非零元素: 5,682,268
🔨 构建 BM25...
💾 保存索引...

✅ 索引构建完成！
   FAISS HNSW: 66,259 条
   Sparse CSR: (66259, 249984), nnz=5,682,268
   BM25: 66,259 条


In [ ]:
# ============================================================
# Cell 6: RAG 检索系统（CSR矩阵版Sparse检索）
# ============================================================
import pickle
import numpy as np
import faiss
import jieba
import torch
import scipy.sparse as sp
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import CrossEncoder
from openai import OpenAI
from typing import List, Dict


class LiteratureRAG:

    def __init__(self, index_dir: str = INDEX_DIR):
        print('🔄 初始化 RAG 系统...')
        index_dir = Path(index_dir)

        self.faiss_index = faiss.read_index(str(index_dir / 'hnsw.index'))
        # CSR矩阵：直接内存映射加载，大矩阵也很快
        self.sparse_matrix = sp.load_npz(str(index_dir / 'sparse.npz'))  # (n, vocab)
        with open(index_dir / 'vocab_size.txt') as f:
            self.vocab_size = int(f.read().strip())
        with open(index_dir / 'bm25.pkl', 'rb') as f:
            self.bm25 = pickle.load(f)
        with open(index_dir / 'metadata.pkl', 'rb') as f:
            self.metadata = pickle.load(f)

        self.embed_model = BGEM3FlagModel(
            EMBED_MODEL, use_fp16=True,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.reranker = CrossEncoder(
            RERANK_MODEL,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.llm = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)

        print(f'✅ 初始化完成，索引 {len(self.metadata):,} 条')
        print(f'   Sparse矩阵: {self.sparse_matrix.shape}')

    def _encode_query(self, query: str) -> Dict:
        out = self.embed_model.encode(
            [query], return_dense=True, return_sparse=True, return_colbert_vecs=False
        )
        dense = np.array(out['dense_vecs'], dtype='float32')
        faiss.normalize_L2(dense)
        return {'dense': dense, 'sparse': out['lexical_weights'][0]}

    def _search_dense(self, query_dense: np.ndarray, top_k: int) -> List[int]:
        _, indices = self.faiss_index.search(query_dense, top_k)
        return [int(i) for i in indices[0] if i >= 0]

    def _search_sparse(self, query_sparse: Dict, top_k: int) -> List[int]:
        """
        CSR矩阵版稀疏检索。
        把query_sparse转为稀疏行向量，一次矩阵乘法得到全部分数。
        30万条数据耗时约 5-20ms（vs List[dict]遍历的 2000-5000ms）。
        """
        # 将query sparse dict 转为 (1, vocab_size) CSR行向量
        q_cols = []
        q_vals = []
        for token_id, weight in query_sparse.items():
            tid = int(token_id)
            if tid < self.vocab_size:  # 防止OOV token越界
                q_cols.append(tid)
                q_vals.append(float(weight))

        if not q_cols:
            return list(range(min(top_k, len(self.metadata))))

        query_vec = sp.csr_matrix(
            ([1.0] * len(q_vals), ([0] * len(q_cols), q_cols)),  # 用1.0占位
            shape=(1, self.vocab_size), dtype=np.float32
        )
        # 真正的加权：手动乘权重
        query_vec = sp.csr_matrix(
            (q_vals, ([0] * len(q_cols), q_cols)),
            shape=(1, self.vocab_size), dtype=np.float32
        )

        # 核心：一次矩阵乘法，结果形状 (1, n_chunks)
        scores = (query_vec @ self.sparse_matrix.T).toarray().flatten()

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [int(i) for i in top_indices]

    def _search_bm25(self, query: str, top_k: int) -> List[int]:
        tokens = list(jieba.cut(query))
        expanded = []
        for t in tokens:
            if re.match(r'[a-zA-Z]+', t):
                expanded.extend(t.lower().split())
            else:
                expanded.append(t)
        scores = self.bm25.get_scores(expanded)
        return [int(i) for i in np.argsort(scores)[::-1][:top_k]]

    @staticmethod
    def _rrf_fusion(rankings: List[List[int]], k: int = RRF_K) -> List[int]:
        scores = {}
        for ranking in rankings:
            for rank, idx in enumerate(ranking):
                scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    def _rerank(self, query: str, doc_indices: List[int], top_k: int) -> List[Dict]:
        candidates = [self.metadata[i] for i in doc_indices]
        scores = self.reranker.predict([[query, c['content'][:512]] for c in candidates])
        results = [{**m, 'rerank_score': float(s)} for m, s in zip(candidates, scores)]
        results.sort(key=lambda x: x['rerank_score'], reverse=True)
        return results[:top_k]

    def retrieve(self, query: str, top_k: int = TOP_K_RERANK,
                 top_k_retrieve: int = TOP_K_RETRIEVE) -> List[Dict]:
        q = self._encode_query(query)
        fused = self._rrf_fusion([
            self._search_dense(q['dense'], top_k_retrieve),
            self._search_sparse(q['sparse'], top_k_retrieve),
            self._search_bm25(query, top_k_retrieve),
        ])
        return self._rerank(query, fused[:top_k * 3], top_k)

    def ask(self, query: str, top_k: int = TOP_K_RERANK,
            system_prompt: str = None, temperature: float = 0.3) -> Dict:
        sources = self.retrieve(query, top_k=top_k)
        context = "\n\n---\n\n".join(
            f"[来源{i+1}] {s['doc_id']} | "
            f"{'|'.join(v for v in s['headers'].values() if v)}\n{s['content'][:800]}"
            for i, s in enumerate(sources)
        )
        if system_prompt is None:
            system_prompt = (
                "你是严谨的科研助手，专注文献和专利分析。"
                "基于检索内容回答，明确标注来源编号，不得捏造。"
            )
        resp = self.llm.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f"文献片段：\n{context}\n\n问题：{query}"}
            ],
            max_tokens=2048, temperature=temperature,
        )
        return {'answer': resp.choices[0].message.content, 'sources': sources, 'query': query}

    def debug_retrieve(self, query: str, top_k: int = 5):
        """打印三路召回对比，用于调参"""
        q = self._encode_query(query)
        dense_r  = self._search_dense(q['dense'], top_k)
        sparse_r = self._search_sparse(q['sparse'], top_k)
        bm25_r   = self._search_bm25(query, top_k)
        print(f'\n🔍 {query}')
        for label, r in [('Dense', dense_r), ('Sparse', sparse_r), ('BM25', bm25_r)]:
            print(f'\n【{label}】')
            for i, idx in enumerate(r, 1):
                m = self.metadata[idx]
                print(f'  {i}. [{m["doc_id"][:25]}] {m["content"][:70]}...')
        fused = self._rrf_fusion([dense_r, sparse_r, bm25_r])
        print('\n【RRF融合 Top5】')
        for i, idx in enumerate(fused[:top_k], 1):
            m = self.metadata[idx]
            print(f'  {i}. [{m["doc_id"][:25]}] {m["content"][:70]}...')

In [ ]:
# ============================================================
# Cell 7: 评估体系（RAGAS框架）
# ============================================================
# RAGAS 四维指标：
#   context_recall    - 检索内容是否覆盖了答案所需信息（需要ground truth）
#   context_precision - 检索结果中有多少真正有用（噪声比）
#   faithfulness      - 生成答案是否忠实于检索内容（幻觉检测）
#   answer_relevancy  - 答案是否真正回答了问题
#
# 使用流程：
#   Step A: 从你的文献库中自动生成测试问答对（用LLM，无需手动标注）
#   Step B: 跑RAG系统获取答案和检索内容
#   Step C: 用RAGAS计算四个指标
#   Step D: 输出评估报告，定位问题

import json
import random
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_recall, context_precision,
    faithfulness, answer_relevancy
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


# ---- Step A: 自动生成测试集 ----

def generate_test_questions(
    rag: LiteratureRAG,
    n_questions: int = 50,
    save_path: str = None
) -> List[Dict]:
    """
    从文献库中采样chunk，用LLM自动生成问答对作为测试集。
    生成的问题覆盖：事实性问题、比较性问题、机理性问题。

    Returns:
        [{"question": ..., "ground_truth": ..., "source_chunk": ...}]
    """
    llm = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)

    # 随机采样文本chunk（排除图片）
    text_chunks = [m for m in rag.metadata if m['type'] == 'text']
    sampled = random.sample(text_chunks, min(n_questions * 3, len(text_chunks)))

    qa_pairs = []
    print(f'🔄 生成测试问答对（目标: {n_questions} 条）...')

    for chunk in sampled:
        if len(qa_pairs) >= n_questions:
            break
        if len(chunk['content']) < 100:
            continue

        prompt = f"""以下是一段来自化学/材料科学文献的内容：

{chunk['content'][:600]}

请基于此段内容生成1个高质量的测试问题，要求：
1. 问题必须能从上文中找到明确答案
2. 问题类型可以是：事实查询、原理解释、方法比较
3. 避免过于简单的是/否问题

请用JSON格式输出，只输出JSON，不要其他文字：
{{"question": "问题内容", "ground_truth": "标准答案（2-4句话）"}}"""

        try:
            resp = llm.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=300, temperature=0.7
            )
            raw = resp.choices[0].message.content.strip()
            # 清理可能的markdown代码块
            raw = re.sub(r'^```json\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()
            qa = json.loads(raw)
            qa['source_chunk'] = chunk['content'][:300]
            qa['source_doc']   = chunk['doc_id']
            qa_pairs.append(qa)
        except Exception as e:
            continue  # 解析失败跳过，不中断

    print(f'✅ 生成测试集: {len(qa_pairs)} 条')

    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
        print(f'💾 测试集保存至: {save_path}')

    return qa_pairs


# ---- Step B+C: 运行评估 ----

def run_ragas_evaluation(
    rag: LiteratureRAG,
    test_questions: List[Dict],
    top_k: int = 5,
    save_path: str = None
) -> Dict:
    """
    运行RAGAS四维评估。

    Args:
        test_questions: generate_test_questions()的输出
        top_k: 每次检索返回的文档数

    Returns:
        scores: {metric_name: score}
    """
    print(f'🔄 开始评估 {len(test_questions)} 条问题...')

    questions, answers, contexts, ground_truths = [], [], [], []

    for i, qa in enumerate(test_questions):
        print(f'  [{i+1}/{len(test_questions)}] {qa["question"][:50]}...')
        result = rag.ask(qa['question'], top_k=top_k)

        questions.append(qa['question'])
        answers.append(result['answer'])
        contexts.append([s['content'] for s in result['sources']])
        ground_truths.append(qa['ground_truth'])

    # 构建RAGAS数据集格式
    dataset = Dataset.from_dict({
        'question':     questions,
        'answer':       answers,
        'contexts':     contexts,
        'ground_truth': ground_truths,
    })

    # RAGAS需要一个LLM来做评判（用DeepSeek包装成OpenAI接口）
    judge_llm = ChatOpenAI(
        model=LLM_MODEL,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
    )
    judge_emb = OpenAIEmbeddings(
        model="text-embedding-3-small",  # 仅answer_relevancy指标需要embedding
        api_key=LLM_API_KEY,             # 如果DeepSeek不支持embedding，换成OpenAI Key
    )

    scores = evaluate(
        dataset,
        metrics=[context_recall, context_precision, faithfulness, answer_relevancy],
        llm=judge_llm,
        embeddings=judge_emb,
    )

    result_dict = scores.to_pandas().mean().to_dict()

    # ---- Step D: 打印评估报告 ----
    print('\n' + '='*60)
    print('📊 RAGAS 评估报告')
    print('='*60)
    metrics_info = {
        'context_recall':    ('检索召回率',  '检索内容是否覆盖答案所需信息'),
        'context_precision': ('检索精准率',  '检索结果中噪声比例'),
        'faithfulness':      ('答案忠实度',  '生成答案是否基于检索内容（幻觉检测）'),
        'answer_relevancy':  ('答案相关性',  '答案是否真正回答了问题'),
    }
    for key, (cn_name, desc) in metrics_info.items():
        val = result_dict.get(key, 0)
        bar = '█' * int(val * 20) + '░' * (20 - int(val * 20))
        status = '✅' if val >= 0.7 else ('⚠️' if val >= 0.5 else '❌')
        print(f'{status} {cn_name:8s} [{bar}] {val:.3f}')
        print(f'   {desc}')

    print('\n💡 问题诊断建议:')
    if result_dict.get('context_recall', 1) < 0.6:
        print('  ❌ 检索召回率低 → 考虑增大 TOP_K_RETRIEVE，或检查chunking是否切太碎')
    if result_dict.get('context_precision', 1) < 0.6:
        print('  ❌ 检索精准率低 → 噪声太多，考虑调高 Reranker 阈值，或改善embedding模型')
    if result_dict.get('faithfulness', 1) < 0.7:
        print('  ❌ 答案忠实度低 → LLM产生幻觉，考虑降低temperature或加强system_prompt约束')
    if result_dict.get('answer_relevancy', 1) < 0.7:
        print('  ❌ 答案相关性低 → 答案未切题，检查prompt设计或query理解')

    if save_path:
        detailed = scores.to_pandas()
        detailed.to_csv(save_path, index=False, encoding='utf-8')
        print(f'\n💾 详细结果保存至: {save_path}')

    return result_dict


# ---- 快速评估（不需要ground_truth，只看faithfulness）----

def quick_faithfulness_check(rag: LiteratureRAG, queries: List[str]) -> None:
    """
    不需要标注数据的快速幻觉检测。
    原理：让LLM判断答案中每句话是否能在检索内容中找到支撑。
    """
    llm = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
    print('🔍 快速幻觉检测（无需标注数据）')
    print('='*60)

    for query in queries:
        result = rag.ask(query)
        context = '\n'.join(s['content'][:300] for s in result['sources'])

        judge_prompt = f"""判断以下「回答」中的每个陈述是否能在「检索内容」中找到直接支撑。

检索内容：
{context}

回答：
{result['answer']}

请逐句分析，输出JSON：
{{"supported": ["有支撑的句子"], "unsupported": ["无支撑的句子（可能是幻觉）"], "score": 0.0-1.0}}"""

        resp = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role': 'user', 'content': judge_prompt}],
            max_tokens=500, temperature=0
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```json\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()

        try:
            judgment = json.loads(raw)
            score = judgment.get('score', 0)
            status = '✅' if score >= 0.8 else ('⚠️' if score >= 0.6 else '❌')
            print(f'{status} [{score:.2f}] {query[:50]}...')
            if judgment.get('unsupported'):
                print(f'  ⚠️  疑似幻觉: {judgment["unsupported"][:2]}')
        except:
            print(f'❓ 解析失败: {query[:50]}...')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_16987/3461534475.py:20: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_16987/3461534475.py:20: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metr

In [ ]:
# ============================================================
# Cell 8: 完整使用流程
# ============================================================

# ---- 第一次：建索引 ----
# all_chunks = load_all_documents(MINERU_DIR, use_vlm=True)
# build_index(all_chunks, INDEX_DIR)

# ---- 日常使用：加载索引 ----
rag = LiteratureRAG(index_dir=INDEX_DIR)

# ---- 检索 + 生成 ----
# result = rag.ask("Which photoinitiators work better under LED light sources?")
# print(result['answer'])

# ---- 评估：自动生成测试集 ----
test_qs = generate_test_questions(
    rag, n_questions=50,
    save_path=f"{EVAL_DIR}/test_questions.json"
)

# ---- 评估：运行RAGAS ----
# scores = run_ragas_evaluation(
#     rag, test_qs, top_k=5,
#     save_path=f"{EVAL_DIR}/ragas_results.csv"
# )

# ---- 快速幻觉检测（不需要标注数据）----
# quick_faithfulness_check(rag, [
#     "LED光源下效果最好的光引发剂是什么？",
#     "自由基光引发剂和阳离子光引发剂有什么区别？",
#     "Which photoinitiators have absorption above 400nm?"
# ])

🔄 初始化 RAG 系统...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

✅ 初始化完成，索引 66,259 条
   Sparse矩阵: (66259, 249984)
🔄 生成测试问答对（目标: 50 条）...
✅ 生成测试集: 50 条
💾 测试集保存至: /content/drive/MyDrive/PatentRAG_v3/eval/test_questions.json


## 文献专利 RAG 系统 v4

### v4 相对 v3 的改动
| 模块 | v3 | v4 |
|------|----|--------|
| 文档解析 | 仅用 `full.md` + 正则提取 | **`content_list.json` 结构化解析**：按 block 类型(text/table/image/equation)差异化处理，保留页码、层级 |
| 文档树 | 无 | **层级文档树**：借鉴 PageIndex 思想，从 MinerU layout 构建 section 树，支持上下文扩展 |
| 表格处理 | 当文本切块（HTML混杂） | **表格三重表示**：结构化行文本(BM25) + LLM摘要(Dense) + 原始HTML(生成时) |
| 图片处理 | 分级VLM | **多粒度 + Late Binding**：caption + VLM描述 + 周围引用 合并检索；生成时注入原始图片 |
| 切块策略 | 统一 1536 字符 | **Parent-Child Chunking**：小chunk精确检索 → 回溯parent大chunk送LLM |
| 元数据 | 仅 doc_id + headers | **丰富元数据**：DOI、作者、年份、文档类型、页码，支持 metadata filtering |
| 数据清洗 | 无 | **文档分类过滤**：自动识别并排除非学术文档（发票、收据等） |
| 检索增强 | 三路RRF | 三路RRF + **上下文扩展**（命中chunk后沿文档树取parent section） |
| 生成增强 | 纯文本上下文 | **多模态生成**：文本 + 原始表格 + 原始图片 一起送多模态LLM |

In [ ]:
# ============================================================
# Cell 1: 安装依赖
# ============================================================

# 强制清除所有相关包，确保没有版本残留
!pip uninstall -y FlagEmbedding transformers -q
!pip install "transformers==4.46.3" -q
!pip install "FlagEmbedding==1.3.5" -q
!pip install faiss-cpu sentence-transformers rank_bm25 jieba openai tqdm scipy -q
!pip install langchain langchain-text-splitters langchain-openai ragas datasets pillow -q

print("✅ 安装完成，现在点击菜单：Runtime → Restart session")
print("⚠️  必须重启！不重启的话import还是读旧版本")

In [ ]:
# ============================================================
# Cell 2: 配置
# ============================================================
import os
from pathlib import Path

GDRIVE_BASE    = "/content/drive/MyDrive"
MINERU_DIR     = f"{GDRIVE_BASE}/MinerU"
PROJECT_DIR    = f"{GDRIVE_BASE}/PatentRAG_v4"
INDEX_DIR      = f"{PROJECT_DIR}/index"
EVAL_DIR       = f"{PROJECT_DIR}/eval"
TREE_DIR       = f"{PROJECT_DIR}/trees"   # 文档树缓存
os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,  exist_ok=True)
os.makedirs(TREE_DIR,  exist_ok=True)

from google.colab import userdata

# ---- 模型 ----
EMBED_MODEL    = "BAAI/bge-m3"
RERANK_MODEL   = "BAAI/bge-reranker-v2-m3"

# ---- LLM（DeepSeek 用于生成；也用于评估）----
LLM_API_KEY    = userdata.get('deepseek_api_key') # 从 Colab Secrets 获取密钥
LLM_BASE_URL   = "https://api.deepseek.com/v1"
LLM_MODEL      = "deepseek-chat"

# ---- 多模态：用于化学图片描述（调API，无需本地GPU）----
# 建议用 GPT-4o 或 Claude，它们对化学图的理解最好
# 如果不想用，设为 None，图片退回纯caption文本处理
VLM_API_KEY    = userdata.get('kimi_api_key') # 从 Colab Secrets 获取密钥
VLM_PROVIDER   = "kimi"
VLM_BASE_URL = "https://api.moonshot.cn/v1"
VLM_MODEL      = "kimi-k2.5"

# ---- 检索参数 ----
CHUNK_SIZE     = 512             # 子chunk大小（字符）
CHUNK_OVERLAP  = 64
PARENT_CHUNK_SIZE = 2048         # 父chunk大小（字符），用于上下文扩展
TOP_K_RETRIEVE = 30
TOP_K_RERANK   = 5
RRF_K          = 60

# ---- 图片处理 ----
CAPTION_SUFFICIENT_LEN = 50

# ---- 表格处理 ----
TABLE_SUMMARY_ENABLED = True     # 是否用LLM为表格生成摘要
TABLE_MAX_HTML_LEN    = 5000     # 超长表格截断

print(f'\u2705 配置完成 | 项目目录: {PROJECT_DIR}')



✅ 配置完成 | 项目目录: /content/drive/MyDrive/PatentRAG_v4


In [ ]:
# ============================================================
# Cell 3: VLM 图片处理（沿用v3，微调）
# ============================================================
import base64
import re
from pathlib import Path
from openai import OpenAI


def _init_vlm_client():
    """初始化VLM客户端"""
    if VLM_PROVIDER == "kimi":
        # Kimi 完全兼容 OpenAI SDK，只需改 base_url
        return OpenAI(api_key=VLM_API_KEY, base_url=VLM_BASE_URL)
    return None

def _init_llm_client():
    """初始化LLM客户端（用于表格摘要等）"""
    if LLM_API_KEY:
        return OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
    return None

_CHEM_IMAGE_KEYWORDS = [
    'mechanism', 'reaction', 'scheme', '机理', '反应路径', '合成路线',
    'structure', '结构式', 'nmr', 'ir', 'uv', 'spectrum', '光谱',
    'cycle', 'pathway', '循环', '路径', 'figure', '图',
    'catalyst', 'catalytic', '催化', 'synthesis', '合成',
]


def _needs_vlm(caption: str, surrounding_text: str) -> bool:
    if VLM_PROVIDER is None:
        return False
    if len(caption) >= CAPTION_SUFFICIENT_LEN:
        return False
    context = (caption + ' ' + surrounding_text).lower()
    return any(kw in context for kw in _CHEM_IMAGE_KEYWORDS)


def _encode_image_b64(image_path: str) -> tuple:
    """读取图片并返回 (base64_str, media_type)"""
    with open(image_path, 'rb') as f:
        img_b64 = base64.b64encode(f.read()).decode()
    ext = Path(image_path).suffix.lower()
    media_type = {
        '.png': 'image/png', '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg', '.gif': 'image/gif',
        '.webp': 'image/webp'
    }.get(ext, 'image/png')
    return img_b64, media_type


def describe_image_with_vlm(image_path: str, caption: str, vlm_client) -> str:
    """
    调用 Kimi Vision API 生成化学图片的详细文字描述。
    注意：Kimi Vision 只支持 base64，不支持图片URL。
    """
    try:
        with open(image_path, 'rb') as f:
            img_b64 = base64.b64encode(f.read()).decode()

        ext = Path(image_path).suffix.lower()
        media_type = {
            '.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.gif': 'image/gif',
            '.webp': 'image/webp'
        }.get(ext, 'image/png')

        prompt = (
            f"这是一张来自化学/材料科学文献的图片，原始标注为：'{caption}'。\n"
            "请详细描述图中的内容，重点关注：\n"
            "1. 如果是反应机理图：描述反应步骤、中间体、箭头方向\n"
            "2. 如果是分子结构式：描述官能团、连接方式\n"
            "3. 如果是光谱图：描述峰位、形状特征\n"
            "4. 如果是实验装置/流程图：描述各组成部分\n"
            "请用中文输出，100-200字。"
        )

        # Kimi Vision：base64图片通过 image_url 的 data URI 传入
        resp = vlm_client.chat.completions.create(
            model=VLM_MODEL,  # kimi-k2.5
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            # Kimi 只支持 base64，格式：data:{mime};base64,{data}
                            "url": f"data:{media_type};base64,{img_b64}"
                        }
                    },
                    {"type": "text", "text": prompt}
                ]
            }],
            max_tokens=300,
        )
        return resp.choices[0].message.content.strip()

    except Exception as e:
        print(f'  ⚠️  Kimi Vision 失败 {Path(image_path).name}: {e}，退回caption')
        return caption


print('\u2705 VLM 模块就绪')


✅ VLM 模块就绪


In [ ]:
# ============================================================
# Cell 4: 表格处理模块（v4 新增）
# ============================================================
from bs4 import BeautifulSoup
from typing import List, Dict, Optional


def html_table_to_row_text(html: str) -> str:
    """
    将HTML表格转为「列头: 值」的逐行文本。
    用于BM25精确匹配——去除HTML标签噪声，保留数据关联。

    例：
      Entry | Catalyst | Yield
      1     | Pd/C     | 95%
    → "Entry: 1 | Catalyst: Pd/C | Yield: 95%"
    """
    soup = BeautifulSoup(html, 'lxml')
    table = soup.find('table')
    if not table:
        # 退回纯文本
        return soup.get_text(separator=' ', strip=True)

    rows = table.find_all('tr')
    if not rows:
        return soup.get_text(separator=' ', strip=True)

    # 提取表头（第一行）
    header_cells = rows[0].find_all(['th', 'td'])
    headers = [c.get_text(strip=True) for c in header_cells]

    lines = []
    for row in rows[1:]:
        cells = row.find_all(['th', 'td'])
        vals = [c.get_text(strip=True) for c in cells]
        # 对齐列头和值
        parts = []
        for j, val in enumerate(vals):
            if not val:
                continue
            hdr = headers[j] if j < len(headers) else f'Col{j+1}'
            if hdr and hdr != val:
                parts.append(f'{hdr}: {val}')
            else:
                parts.append(val)
        if parts:
            lines.append(' | '.join(parts))

    if not lines:
        # 表头和数据区分不清，退回纯文本
        return soup.get_text(separator=' ', strip=True)

    header_line = ' | '.join(h for h in headers if h)
    return f"表头: {header_line}\n" + '\n'.join(lines)


def summarize_table_with_llm(table_text: str, caption: str,
                              breadcrumb: str, llm_client) -> str:
    """
    用LLM为表格生成自然语言摘要（用于Dense语义匹配）。
    """
    prompt = (
        f"以下是一张来自化学/材料科学文献的表格。\n"
        f"表格标题: {caption}\n"
        f"所在章节: {breadcrumb}\n\n"
        f"表格内容:\n{table_text[:2000]}\n\n"
        f"请用一段话（100-200字）概括这张表格的关键信息，"
        f"包括：表格比较了什么、关键发现、最优条件/结果。"
    )
    try:
        resp = llm_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=300, temperature=0.3,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f'  \u26a0\ufe0f  表格摘要生成失败: {e}')
        return table_text[:300]


print('\u2705 表格处理模块就绪')

✅ 表格处理模块就绪


In [ ]:
# ============================================================
# Cell 5: 文档树构建 + 元数据提取（v4 核心新增）
# ============================================================
import json
from typing import List, Dict, Optional, Tuple


def extract_metadata_from_doc(blocks: List[Dict], doc_dir_name: str) -> Dict:
    """
    从 content_list.json 的前几个 block 提取文献元数据。
    返回: {doi, title, authors, doc_type, ...}
    """
    meta = {
        'doc_id': doc_dir_name,
        'title': '',
        'authors': '',
        'doi': '',
        'doc_type': 'unknown',  # paper | patent | other
    }

    # 从目录名提取DOI（MinerU文件夹常以DOI命名）
    doi_match = re.search(r'(10\.\d{4,}[/@][^\s]+)', doc_dir_name.replace('@', '/'))
    if doi_match:
        meta['doi'] = doi_match.group(1)

    # 扫描前15个block
    header_texts = []
    all_early_text = []
    for block in blocks[:15]:
        if block['type'] == 'text':
            txt = block.get('text', '').strip()
            if not txt:
                continue
            all_early_text.append(txt)
            level = block.get('text_level', 0)
            if level == 1 and not meta['title']:
                meta['title'] = txt

    # 从前几行猜测作者（通常在标题之后、Abstract之前）
    found_title = False
    for txt in all_early_text:
        if txt == meta['title']:
            found_title = True
            continue
        if found_title and 'abstract' not in txt.lower():
            # 短文本、含有 *, and 等，可能是作者行
            if len(txt) < 300 and ('*' in txt or ' and ' in txt.lower() or ',' in txt):
                meta['authors'] = txt
                break
        if 'abstract' in txt.lower():
            break

    # 文档类型分类
    full_text = ' '.join(all_early_text).lower()
    academic_signals = ['abstract', 'introduction', 'doi', 'references',
                        'keywords', 'journal', 'received', 'accepted']
    patent_signals = ['patent', '专利', 'claim', '权利要求', 'applicant', '申请人']
    noise_signals = ['发票', 'invoice', '增值税', '收据', 'receipt']

    if any(sig in full_text for sig in noise_signals):
        meta['doc_type'] = 'noise'
    elif any(sig in full_text for sig in patent_signals):
        meta['doc_type'] = 'patent'
    elif any(sig in full_text for sig in academic_signals):
        meta['doc_type'] = 'paper'
    else:
        meta['doc_type'] = 'unknown'

    return meta


def build_document_tree(blocks: List[Dict], doc_meta: Dict) -> Dict:
    """
    从 content_list.json 构建文档层级树。
    借鉴 PageIndex 思想：保留文档自然结构，不强制切块。

    树结构:
    {
      'meta': {...},
      'sections': [
        {
          'title': 'Introduction',
          'level': 2,
          'page_idx': 0,
          'blocks': [...],        # 该section下的原始blocks
          'full_text': '...',     # 拼接后的完整section文本
          'children': [...]       # 子section
        },
        ...
      ]
    }
    """
    tree = {'meta': doc_meta, 'sections': []}

    # 将blocks按标题分段
    # text_level > 0 的block视为section标题
    current_section = {
        'title': doc_meta.get('title', 'Untitled'),
        'level': 0,
        'page_idx': 0,
        'blocks': [],
        'children': [],
    }

    # 用栈维护层级
    section_stack = [current_section]

    for block in blocks:
        is_heading = (block['type'] == 'text'
                      and block.get('text_level', 0) > 0
                      and len(block.get('text', '')) < 200)

        if is_heading:
            level = block['text_level']
            new_section = {
                'title': block['text'].strip(),
                'level': level,
                'page_idx': block.get('page_idx', 0),
                'blocks': [],
                'children': [],
            }
            # 回溯到合适的父节点
            while len(section_stack) > 1 and section_stack[-1]['level'] >= level:
                section_stack.pop()
            section_stack[-1]['children'].append(new_section)
            section_stack.append(new_section)
        else:
            # 内容block归入当前section
            section_stack[-1]['blocks'].append(block)

    tree['sections'] = current_section['children'] if current_section['children'] else [current_section]
    # 顶层未分配的blocks保留在root
    tree['root_blocks'] = current_section['blocks']

    # 为每个section生成full_text
    _compute_section_text(tree['sections'])
    _compute_section_text([{'blocks': tree['root_blocks'], 'children': [], 'title': '', 'level': 0}])

    return tree


def _compute_section_text(sections: List[Dict]):
    """递归为每个section拼接full_text"""
    for sec in sections:
        parts = []
        for b in sec.get('blocks', []):
            if b['type'] == 'text':
                txt = b.get('text', '').strip()
                if txt:
                    parts.append(txt)
            elif b['type'] == 'table':
                cap = ' '.join(b.get('table_caption', []))
                parts.append(f'[TABLE: {cap}]' if cap else '[TABLE]')
            elif b['type'] == 'image':
                cap = ' '.join(b.get('image_caption', []))
                parts.append(f'[FIGURE: {cap}]' if cap else '[FIGURE]')
        sec['full_text'] = '\n'.join(parts)
        _compute_section_text(sec.get('children', []))


def get_breadcrumb(section_path: List[Dict]) -> str:
    """从section路径生成面包屑"""
    return ' > '.join(s['title'] for s in section_path if s.get('title'))


def flatten_sections(tree: Dict) -> List[Tuple[List[Dict], Dict]]:
    """
    展平文档树，返回 [(path, section), ...]。
    path 是从根到该section的路径，用于生成breadcrumb和parent回溯。
    """
    results = []
    def _walk(sections, path):
        for sec in sections:
            current_path = path + [sec]
            results.append((current_path, sec))
            _walk(sec.get('children', []), current_path)
    _walk(tree.get('sections', []), [])
    return results


print('\u2705 文档树构建模块就绪')

✅ 文档树构建模块就绪


In [ ]:
# ============================================================
# Cell 6: 结构化解析 + Parent-Child Chunking（v4 核心）
# ============================================================
import jieba
from langchain_text_splitters import RecursiveCharacterTextSplitter


def parse_mineru_doc_v4(
    doc_dir: Path,
    vlm_client=None,
    llm_client=None,
) -> Tuple[List[Dict], Optional[Dict]]:
    """
    v4 解析：基于 content_list.json 结构化解析。

    返回:
        chunks: List[Dict] - 所有chunk
        tree:   Dict       - 文档树（用于上下文扩展）
    """
    doc_id = doc_dir.name

    # ---- 1. 寻找 content_list.json ----
    content_list_files = list(doc_dir.glob('*_content_list.json'))
    if not content_list_files:
        # 退回v3方式（用full.md）
        return _parse_fallback_fullmd(doc_dir, vlm_client), None

    with open(content_list_files[0], 'r', encoding='utf-8') as f:
        blocks = json.load(f)

    if not blocks:
        return [], None

    # ---- 2. 元数据提取 + 文档分类 ----
    doc_meta = extract_metadata_from_doc(blocks, doc_id)

    if doc_meta['doc_type'] == 'noise':
        print(f'  \u26d4 跳过非学术文档: {doc_id[:50]}... (检测为: {doc_meta["doc_type"]})')
        return [], None

    # ---- 3. 构建文档树 ----
    tree = build_document_tree(blocks, doc_meta)

    # ---- 4. 按block类型差异化处理 ----
    chunks = []
    flat_sections = flatten_sections(tree)

    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=['\n\n', '\n', '。', '；', '. ', ' ', '']
    )

    for path, section in flat_sections:
        breadcrumb = get_breadcrumb(path)
        # 父级文本（用于 parent-child 上下文扩展）
        parent_text = section.get('full_text', '')[:PARENT_CHUNK_SIZE]
        page_idx = section.get('page_idx', 0)

        for block in section.get('blocks', []):
            b_type = block['type']
            b_page = block.get('page_idx', page_idx)

            # ---- TEXT blocks ----
            if b_type == 'text':
                txt = block.get('text', '').strip()
                if len(txt) < 30:
                    continue

                # 子chunk（用于精确检索）
                sub_texts = child_splitter.split_text(txt)
                for st in sub_texts:
                    if len(st.strip()) < 30:
                        continue
                    content = f"{breadcrumb}\n{st}".strip() if breadcrumb else st.strip()
                    chunks.append({
                        'content':          content,
                        'content_for_bm25': ' '.join(jieba.cut(content)),
                        'doc_id':           doc_id,
                        'doc_meta':         doc_meta,
                        'breadcrumb':       breadcrumb,
                        'type':             'text',
                        'page_idx':         b_page,
                        'parent_text':      parent_text,
                        'image_path':       None,
                        'table_html':       None,
                    })

            # ---- TABLE blocks ----
            elif b_type == 'table':
                table_html = block.get('table_body', '')
                table_caption = ' '.join(block.get('table_caption', []))
                table_img = block.get('img_path', '')
                img_abs_path = str(doc_dir / table_img) if table_img else None

                if not table_html:
                    continue

                # 表示1: 结构化行文本（BM25精确匹配）
                row_text = html_table_to_row_text(table_html[:TABLE_MAX_HTML_LEN])
                row_content = f"{breadcrumb}\n[表格] {table_caption}\n{row_text}".strip()

                # 表示2: LLM摘要（Dense语义匹配）
                if TABLE_SUMMARY_ENABLED and llm_client:
                    summary = summarize_table_with_llm(
                        row_text, table_caption, breadcrumb, llm_client
                    )
                    summary_content = f"{breadcrumb}\n[表格摘要] {table_caption}\n{summary}".strip()
                else:
                    summary_content = row_content  # 退回行文本

                # chunk A: 行文本（用于BM25）
                chunks.append({
                    'content':          row_content,
                    'content_for_bm25': ' '.join(jieba.cut(row_content)),
                    'doc_id':           doc_id,
                    'doc_meta':         doc_meta,
                    'breadcrumb':       breadcrumb,
                    'type':             'table',
                    'page_idx':         b_page,
                    'parent_text':      parent_text,
                    'image_path':       img_abs_path,
                    'table_html':       table_html[:TABLE_MAX_HTML_LEN],
                })

                # chunk B: LLM摘要（用于Dense）—— 仅当与行文本不同时才额外添加
                if summary_content != row_content:
                    chunks.append({
                        'content':          summary_content,
                        'content_for_bm25': ' '.join(jieba.cut(summary_content)),
                        'doc_id':           doc_id,
                        'doc_meta':         doc_meta,
                        'breadcrumb':       breadcrumb,
                        'type':             'table_summary',
                        'page_idx':         b_page,
                        'parent_text':      parent_text,
                        'image_path':       img_abs_path,
                        'table_html':       table_html[:TABLE_MAX_HTML_LEN],
                    })

            # ---- IMAGE blocks ----
            elif b_type == 'image':
                img_rel_path = block.get('img_path', '')
                caption_parts = block.get('image_caption', [])
                caption = ' '.join(caption_parts).strip()
                img_abs_path = str(doc_dir / img_rel_path) if img_rel_path else None

                # 周围文本上下文
                surrounding = parent_text[:200] if parent_text else ''

                if not caption:
                    caption = surrounding[:100]

                # VLM描述
                vlm_desc = ''
                if (vlm_client
                    and img_abs_path
                    and Path(img_abs_path).exists()
                    and _needs_vlm(caption, surrounding)):
                    vlm_desc = describe_image_with_vlm(img_abs_path, caption, vlm_client)

                # 合并多粒度文本：caption + VLM描述 + 上下文引用
                content_parts = [f"{breadcrumb}", f"[图片] {caption}"]
                if vlm_desc and vlm_desc != caption:
                    content_parts.append(f"[VLM描述] {vlm_desc}")
                if surrounding and surrounding[:80] not in caption:
                    content_parts.append(f"[上下文] {surrounding[:150]}")
                img_content = '\n'.join(p for p in content_parts if p).strip()

                if len(img_content) > 20:
                    chunks.append({
                        'content':          img_content,
                        'content_for_bm25': ' '.join(jieba.cut(img_content)),
                        'doc_id':           doc_id,
                        'doc_meta':         doc_meta,
                        'breadcrumb':       breadcrumb,
                        'type':             'image',
                        'page_idx':         b_page,
                        'parent_text':      parent_text,
                        'image_path':       img_abs_path,
                        'table_html':       None,
                    })

    return chunks, tree


def _parse_fallback_fullmd(doc_dir: Path, vlm_client=None) -> List[Dict]:
    """
    退回方案：当 content_list.json 不存在时用 full.md 解析（兼容v3）。
    """
    from langchain_text_splitters import MarkdownHeaderTextSplitter

    full_md_path = doc_dir / 'full.md'
    if not full_md_path.exists():
        return []

    raw_md = full_md_path.read_text(encoding='utf-8')
    doc_id = doc_dir.name
    chunks = []

    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ('#', 'Header1'), ('##', 'Header2'), ('###', 'Header3')
        ],
        strip_headers=False
    )
    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=['\n\n', '\n', '。', '；', '. ', ' ', '']
    )

    doc_meta = {
        'doc_id': doc_id, 'title': '', 'authors': '',
        'doi': '', 'doc_type': 'unknown'
    }
    doi_match = re.search(r'(10\.\d{4,}[/@][^\s]+)', doc_id.replace('@', '/'))
    if doi_match:
        doc_meta['doi'] = doi_match.group(1)

    for section in header_splitter.split_text(raw_md):
        headers = section.metadata
        text = section.page_content
        breadcrumb = ' > '.join(v for v in headers.values() if v)

        # 图片处理（正则，与v3兼容）
        img_pattern = r'!\[(.*?)\]\((images/[^)]+)\)'
        for img_match in re.finditer(img_pattern, text):
            caption = img_match.group(1).strip()
            img_path = str(doc_dir / img_match.group(2))
            pos = img_match.start()
            surrounding = text[max(0, pos-150):pos+150].strip()
            if not caption:
                caption = surrounding[:100]

            if vlm_client and _needs_vlm(caption, surrounding) and Path(img_path).exists():
                description = describe_image_with_vlm(img_path, caption, vlm_client)
                img_content = f"[图片|VLM描述] {breadcrumb}\n{description}".strip()
            else:
                img_content = f"[图片] {breadcrumb}\n{caption}".strip()

            if len(img_content) > 20:
                chunks.append({
                    'content': img_content,
                    'content_for_bm25': ' '.join(jieba.cut(img_content)),
                    'doc_id': doc_id, 'doc_meta': doc_meta,
                    'breadcrumb': breadcrumb, 'type': 'image',
                    'page_idx': 0, 'parent_text': '',
                    'image_path': img_path, 'table_html': None,
                })

        clean_text = re.sub(img_pattern, '', text).strip()
        if len(clean_text) < 30:
            continue

        for sc in char_splitter.split_text(clean_text):
            if len(sc.strip()) < 30:
                continue
            final = f"{breadcrumb}\n{sc}".strip() if breadcrumb else sc.strip()
            chunks.append({
                'content': final,
                'content_for_bm25': ' '.join(jieba.cut(final)),
                'doc_id': doc_id, 'doc_meta': doc_meta,
                'breadcrumb': breadcrumb, 'type': 'text',
                'page_idx': 0, 'parent_text': clean_text[:PARENT_CHUNK_SIZE],
                'image_path': None, 'table_html': None,
            })

    return chunks


def load_all_documents_v4(
    mineru_dir: str,
    use_vlm: bool = True,
    use_table_summary: bool = True,
) -> Tuple[List[Dict], Dict[str, Dict]]:
    """
    解析所有文档，返回 (all_chunks, doc_trees)。
    doc_trees: {doc_id: tree_dict} 用于检索时的上下文扩展。
    """
    from tqdm import tqdm

    vlm_client = _init_vlm_client() if use_vlm else None
    llm_client = _init_llm_client() if use_table_summary else None

    status_parts = []
    if vlm_client:
        status_parts.append(f'VLM={VLM_PROVIDER}/{VLM_MODEL}')
    if llm_client:
        status_parts.append(f'表格摘要={LLM_MODEL}')
    print(f'\U0001f4f7 多模态处理: {" | ".join(status_parts) if status_parts else "关闭"}')

    base = Path(mineru_dir)
    all_doc_dirs = sorted(set(f.parent for f in base.rglob('full.md')))
    print(f'\U0001f4c2 发现 {len(all_doc_dirs)} 个文档')

    all_chunks = []
    doc_trees = {}
    stats = {'total': 0, 'paper': 0, 'patent': 0, 'noise_skipped': 0,
             'unknown': 0, 'vlm_images': 0, 'tables': 0, 'table_summaries': 0,
             'fallback_md': 0}

    for doc_dir in tqdm(all_doc_dirs, desc='解析文档'):
        try:
            chunks, tree = parse_mineru_doc_v4(doc_dir, vlm_client, llm_client)

            if not chunks:
                if tree is None:  # 未被过滤，只是空
                    pass
                else:
                    stats['noise_skipped'] += 1
                continue

            # 统计
            stats['total'] += 1
            doc_type = chunks[0].get('doc_meta', {}).get('doc_type', 'unknown')
            if doc_type in stats:
                stats[doc_type] += 1
            stats['vlm_images'] += sum(1 for c in chunks if 'VLM描述' in c['content'])
            stats['tables'] += sum(1 for c in chunks if c['type'] == 'table')
            stats['table_summaries'] += sum(1 for c in chunks if c['type'] == 'table_summary')
            if tree is None:
                stats['fallback_md'] += 1

            all_chunks.extend(chunks)
            if tree:
                doc_trees[doc_dir.name] = tree

        except Exception as e:
            print(f'\u2717 {doc_dir.name[:50]}: {e}')

    print(f'\n\u2705 解析完成！')
    print(f'   文档: {stats["total"]} 篇 (论文:{stats["paper"]}, 专利:{stats["patent"]}, 其他:{stats["unknown"]})')
    print(f'   跳过噪声文档: {stats["noise_skipped"]} 篇')
    print(f'   Chunks 总数: {len(all_chunks):,}')
    print(f'   表格chunks: {stats["tables"]} (含LLM摘要: {stats["table_summaries"]})')
    print(f'   VLM图片: {stats["vlm_images"]} 张')
    print(f'   退回full.md解析: {stats["fallback_md"]} 篇')
    print(f'   文档树: {len(doc_trees)} 棵')

    return all_chunks, doc_trees


print('\u2705 文档解析模块就绪')

✅ 文档解析模块就绪


In [ ]:
# ============================================================
# Cell 7: 构建索引（在v3基础上扩展metadata字段）
# ============================================================
import pickle
import numpy as np
import faiss
import torch
import scipy.sparse as sp
from FlagEmbedding import BGEM3FlagModel
from rank_bm25 import BM25Okapi
from tqdm import tqdm


def build_index_v4(
    all_chunks: List[Dict],
    doc_trees: Dict[str, Dict],
    index_dir: str,
    batch_size: int = 64,
):
    """
    构建三路统一索引 + 文档树持久化。
    索引结构与v3一致（FAISS + CSR + BM25），metadata字段更丰富。
    """
    index_dir = Path(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)

    print('\U0001f504 加载 BGE-M3...')
    model = BGEM3FlagModel(
        EMBED_MODEL, use_fp16=True,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

    contents = [c['content'] for c in all_chunks]
    n = len(contents)
    DENSE_DIM = 1024

    # ------ FAISS HNSW ------
    faiss_index = faiss.IndexHNSWFlat(DENSE_DIM, 32)
    faiss_index.hnsw.efSearch = 64
    faiss_index.hnsw.efConstruction = 64

    # ------ Sparse COO收集 ------
    sparse_rows, sparse_cols, sparse_vals = [], [], []
    max_token_id = 0

    print(f'\U0001f504 批量编码 {n:,} 个chunks...')
    for i in tqdm(range(0, n, batch_size), desc='编码进度'):
        batch = contents[i:i+batch_size]
        outputs = model.encode(
            batch, batch_size=batch_size, max_length=512,
            return_dense=True, return_sparse=True, return_colbert_vecs=False
        )

        dense = np.array(outputs['dense_vecs'], dtype='float32')
        faiss.normalize_L2(dense)
        faiss_index.add(dense)

        for local_j, lex_weights in enumerate(outputs['lexical_weights']):
            global_row = i + local_j
            for token_id, weight in lex_weights.items():
                sparse_rows.append(global_row)
                sparse_cols.append(int(token_id))
                sparse_vals.append(float(weight))
                if int(token_id) > max_token_id:
                    max_token_id = int(token_id)

    # ------ CSR矩阵 ------
    print('\U0001f528 构建 CSR 稀疏矩阵...')
    vocab_size = max_token_id + 1
    sparse_matrix = sp.csr_matrix(
        (sparse_vals, (sparse_rows, sparse_cols)),
        shape=(n, vocab_size), dtype=np.float32
    )
    print(f'   矩阵形状: {sparse_matrix.shape}, 非零元素: {sparse_matrix.nnz:,}')

    # ------ BM25 ------
    print('\U0001f528 构建 BM25...')
    tokenized = [c['content_for_bm25'].split() for c in all_chunks]
    bm25 = BM25Okapi(tokenized)

    # ------ metadata（v4扩展字段）------
    metadata = []
    for c in all_chunks:
        metadata.append({
            'content':      c['content'],
            'doc_id':       c['doc_id'],
            'doc_meta':     c.get('doc_meta', {}),
            'breadcrumb':   c.get('breadcrumb', ''),
            'type':         c['type'],
            'page_idx':     c.get('page_idx', 0),
            'parent_text':  c.get('parent_text', ''),
            'image_path':   c.get('image_path'),
            'table_html':   c.get('table_html'),
        })

    # ------ 保存 ------
    print('\U0001f4be 保存索引...')
    faiss.write_index(faiss_index, str(index_dir / 'hnsw.index'))
    sp.save_npz(str(index_dir / 'sparse.npz'), sparse_matrix)
    with open(index_dir / 'bm25.pkl', 'wb') as f:
        pickle.dump(bm25, f)
    with open(index_dir / 'vocab_size.txt', 'w') as f:
        f.write(str(vocab_size))
    with open(index_dir / 'metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)
    with open(index_dir / 'doc_trees.pkl', 'wb') as f:
        pickle.dump(doc_trees, f)

    print(f'\n\u2705 索引构建完成！')
    print(f'   FAISS HNSW: {faiss_index.ntotal:,} 条')
    print(f'   Sparse CSR: {sparse_matrix.shape}, nnz={sparse_matrix.nnz:,}')
    print(f'   BM25: {len(tokenized):,} 条')
    print(f'   文档树: {len(doc_trees)} 棵')


print('\u2705 索引构建模块就绪')

✅ 索引构建模块就绪


In [ ]:
# ============================================================
# Cell 8: RAG 检索系统 v4
# ============================================================
import pickle
import numpy as np
import faiss
import jieba
import torch
import scipy.sparse as sp
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import CrossEncoder
from openai import OpenAI
from typing import List, Dict, Optional


class LiteratureRAG_v4:

    def __init__(self, index_dir: str = INDEX_DIR):
        print('\U0001f504 初始化 RAG v4 系统...')
        index_dir = Path(index_dir)

        self.faiss_index = faiss.read_index(str(index_dir / 'hnsw.index'))
        self.sparse_matrix = sp.load_npz(str(index_dir / 'sparse.npz'))
        with open(index_dir / 'vocab_size.txt') as f:
            self.vocab_size = int(f.read().strip())
        with open(index_dir / 'bm25.pkl', 'rb') as f:
            self.bm25 = pickle.load(f)
        with open(index_dir / 'metadata.pkl', 'rb') as f:
            self.metadata = pickle.load(f)
        with open(index_dir / 'doc_trees.pkl', 'rb') as f:
            self.doc_trees = pickle.load(f)

        self.embed_model = BGEM3FlagModel(
            EMBED_MODEL, use_fp16=True,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.reranker = CrossEncoder(
            RERANK_MODEL,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.llm = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
        self.vlm_client = _init_vlm_client()

        print(f'\u2705 初始化完成')
        print(f'   索引: {len(self.metadata):,} 条 | Sparse: {self.sparse_matrix.shape}')
        print(f'   文档树: {len(self.doc_trees)} 棵')
        print(f'   多模态生成: {"启用" if self.vlm_client else "关闭"}')

    # ---- 编码 ----
    def _encode_query(self, query: str) -> Dict:
        out = self.embed_model.encode(
            [query], return_dense=True, return_sparse=True, return_colbert_vecs=False
        )
        dense = np.array(out['dense_vecs'], dtype='float32')
        faiss.normalize_L2(dense)
        return {'dense': dense, 'sparse': out['lexical_weights'][0]}

    # ---- 三路检索（与v3一致）----
    def _search_dense(self, query_dense: np.ndarray, top_k: int) -> List[int]:
        _, indices = self.faiss_index.search(query_dense, top_k)
        return [int(i) for i in indices[0] if i >= 0]

    def _search_sparse(self, query_sparse: Dict, top_k: int) -> List[int]:
        q_cols, q_vals = [], []
        for token_id, weight in query_sparse.items():
            tid = int(token_id)
            if tid < self.vocab_size:
                q_cols.append(tid)
                q_vals.append(float(weight))
        if not q_cols:
            return list(range(min(top_k, len(self.metadata))))
        query_vec = sp.csr_matrix(
            (q_vals, ([0] * len(q_cols), q_cols)),
            shape=(1, self.vocab_size), dtype=np.float32
        )
        scores = (query_vec @ self.sparse_matrix.T).toarray().flatten()
        return [int(i) for i in np.argsort(scores)[::-1][:top_k]]

    def _search_bm25(self, query: str, top_k: int) -> List[int]:
        tokens = list(jieba.cut(query))
        expanded = []
        for t in tokens:
            if re.match(r'[a-zA-Z]+', t):
                expanded.extend(t.lower().split())
            else:
                expanded.append(t)
        scores = self.bm25.get_scores(expanded)
        return [int(i) for i in np.argsort(scores)[::-1][:top_k]]

    @staticmethod
    def _rrf_fusion(rankings: List[List[int]], k: int = RRF_K) -> List[int]:
        scores = {}
        for ranking in rankings:
            for rank, idx in enumerate(ranking):
                scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    # ---- Rerank + 上下文扩展 ----
    def _rerank(self, query: str, doc_indices: List[int], top_k: int) -> List[Dict]:
        candidates = [self.metadata[i] for i in doc_indices]
        scores = self.reranker.predict(
            [[query, c['content'][:512]] for c in candidates]
        )
        results = [{**m, 'rerank_score': float(s)} for m, s in zip(candidates, scores)]
        results.sort(key=lambda x: x['rerank_score'], reverse=True)
        return results[:top_k]

    def _expand_context(self, results: List[Dict]) -> List[Dict]:
        """
        v4: 对rerank结果做上下文扩展。
        将命中的子chunk替换为其parent section文本，提供更完整的上下文。
        去重：同一文档同一section的parent不重复。
        """
        expanded = []
        seen_parents = set()

        for r in results:
            parent_text = r.get('parent_text', '')
            # 生成去重key
            parent_key = (r['doc_id'], r.get('breadcrumb', ''), parent_text[:100])

            expanded_item = dict(r)
            if parent_text and parent_key not in seen_parents:
                # 用parent_text替代content作为LLM上下文（更完整）
                expanded_item['expanded_context'] = parent_text
                seen_parents.add(parent_key)
            else:
                expanded_item['expanded_context'] = r['content']

            expanded.append(expanded_item)

        return expanded

    # ---- 检索 ----
    def retrieve(self, query: str, top_k: int = TOP_K_RERANK,
                 top_k_retrieve: int = TOP_K_RETRIEVE) -> List[Dict]:
        q = self._encode_query(query)
        fused = self._rrf_fusion([
            self._search_dense(q['dense'], top_k_retrieve),
            self._search_sparse(q['sparse'], top_k_retrieve),
            self._search_bm25(query, top_k_retrieve),
        ])
        reranked = self._rerank(query, fused[:top_k * 3], top_k)
        return self._expand_context(reranked)

    # ---- 多模态生成 ----
    def _build_multimodal_context(self, sources: List[Dict]) -> tuple:
        """
        构建多模态上下文：文本 + 表格原始HTML + 图片。
        返回 (text_context, image_messages) 用于不同LLM。
        """
        text_parts = []
        image_messages = []  # 用于多模态LLM

        for i, s in enumerate(sources):
            doc_meta = s.get('doc_meta', {})
            doc_label = doc_meta.get('title', s['doc_id'])[:60]
            header = f"[来源{i+1}] {doc_label} | {s.get('breadcrumb', '')} | p.{s.get('page_idx', '?')}"

            # 主要内容：优先使用expanded_context
            main_text = s.get('expanded_context', s['content'])[:1000]

            # 表格：附上原始HTML供LLM直接读表
            if s.get('table_html'):
                # 将HTML转为更易读的文本（LLM能理解HTML表格）
                text_parts.append(f"{header}\n{main_text}\n\n原始表格:\n{s['table_html'][:2000]}")
            else:
                text_parts.append(f"{header}\n{main_text}")

            # 图片：收集用于多模态注入
            if s.get('image_path') and Path(s['image_path']).exists():
                try:
                    img_b64, media_type = _encode_image_b64(s['image_path'])
                    image_messages.append({
                        'source_idx': i + 1,
                        'b64': img_b64,
                        'media_type': media_type,
                        'caption': s.get('breadcrumb', ''),
                    })
                except Exception:
                    pass

        text_context = '\n\n---\n\n'.join(text_parts)
        return text_context, image_messages

    def ask(self, query: str, top_k: int = TOP_K_RERANK,
            system_prompt: str = None, temperature: float = 0.3,
            use_multimodal: bool = True) -> Dict:
        """
        v4: 支持多模态生成。
        当检索结果包含图片且VLM可用时，图片和文本一起送给多模态LLM。
        """
        sources = self.retrieve(query, top_k=top_k)
        text_context, image_messages = self._build_multimodal_context(sources)

        if system_prompt is None:
            system_prompt = (
                "你是严谨的科研助手，专注文献和专利分析。"
                "基于检索内容回答，明确标注来源编号[来源N]，不得捏造。"
                "如果检索内容中包含表格数据，请直接引用表格中的数值。"
                "如果提供了图片，请结合图片内容分析。"
            )

        # 决定是否走多模态路径
        use_vlm_gen = (
            use_multimodal
            and self.vlm_client
            and image_messages
            and VLM_PROVIDER == 'openai'
        )

        if use_vlm_gen:
            # 多模态生成：文本 + 图片一起送GPT-4o
            content_parts = [
                {"type": "text", "text": f"文献片段：\n{text_context}\n\n问题：{query}"}
            ]
            # 最多注入3张图片（控制成本）
            for img in image_messages[:3]:
                content_parts.append({
                    "type": "text",
                    "text": f"\n[来源{img['source_idx']}的图片] {img['caption']}:"
                })
                content_parts.append({
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{img['media_type']};base64,{img['b64']}",
                        "detail": "low"  # 控制token消耗
                    }
                })

            resp = self.vlm_client.chat.completions.create(
                model=VLM_MODEL,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': content_parts}
                ],
                max_tokens=2048, temperature=temperature,
            )
            answer = resp.choices[0].message.content
            gen_mode = 'multimodal'
        else:
            # 纯文本生成（与v3兼容）
            resp = self.llm.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': f"文献片段：\n{text_context}\n\n问题：{query}"}
                ],
                max_tokens=2048, temperature=temperature,
            )
            answer = resp.choices[0].message.content
            gen_mode = 'text'

        return {
            'answer': answer,
            'sources': sources,
            'query': query,
            'gen_mode': gen_mode,
            'n_images': len(image_messages),
            'n_tables': sum(1 for s in sources if s.get('table_html')),
        }

    # ---- 调试工具 ----
    def debug_retrieve(self, query: str, top_k: int = 5):
        """打印三路召回对比 + v4扩展信息"""
        q = self._encode_query(query)
        dense_r  = self._search_dense(q['dense'], top_k)
        sparse_r = self._search_sparse(q['sparse'], top_k)
        bm25_r   = self._search_bm25(query, top_k)

        print(f'\n\U0001f50d {query}')
        for label, r in [('Dense', dense_r), ('Sparse', sparse_r), ('BM25', bm25_r)]:
            print(f'\n\u3010{label}\u3011')
            for i, idx in enumerate(r, 1):
                m = self.metadata[idx]
                t = m['type']
                print(f'  {i}. [{t:6s}] [{m["doc_id"][:25]}] {m["content"][:60]}...')

        fused = self._rrf_fusion([dense_r, sparse_r, bm25_r])
        reranked = self._rerank(query, fused[:top_k * 3], top_k)

        print(f'\n\u3010RRF + Rerank Top{top_k}\u3011')
        for i, r in enumerate(reranked, 1):
            meta = r.get('doc_meta', {})
            print(f'  {i}. [{r["type"]:6s}] score={r["rerank_score"]:.3f}')
            print(f'     doc: {meta.get("title", r["doc_id"])[:50]}')
            print(f'     section: {r.get("breadcrumb", "")} | p.{r.get("page_idx", "?")}')
            print(f'     content: {r["content"][:80]}...')
            if r.get('table_html'):
                print(f'     \u2517 \u6709原始表格HTML')
            if r.get('image_path'):
                print(f'     \u2517 \u6709原始图片: {Path(r["image_path"]).name}')


print('\u2705 RAG v4 系统就绪')

✅ RAG v4 系统就绪


In [ ]:
# ============================================================
# Cell 9: 评估体系（沿用v3 RAGAS，适配v4字段）
# ============================================================
import json
import random
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    context_recall, context_precision,
    faithfulness, answer_relevancy
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


def generate_test_questions(
    rag: LiteratureRAG_v4,
    n_questions: int = 50,
    save_path: str = None,
) -> List[Dict]:
    """
    v4: 从文献库中采样，覆盖 text + table 类型的chunk。
    """
    llm = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)

    # 采样：text + table类型（不只是text）
    eligible = [m for m in rag.metadata
                if m['type'] in ('text', 'table') and len(m['content']) > 100]
    sampled = random.sample(eligible, min(n_questions * 3, len(eligible)))

    qa_pairs = []
    print(f'\U0001f504 生成测试问答对（目标: {n_questions} 条）...')

    for chunk in sampled:
        if len(qa_pairs) >= n_questions:
            break

        chunk_type = '表格' if chunk['type'] == 'table' else '文本'
        prompt = f"""以下是一段来自化学/材料科学文献的{chunk_type}内容：

{chunk['content'][:600]}

请基于此段内容生成1个高质量的测试问题，要求：
1. 问题必须能从上文中找到明确答案
2. 问题类型可以是：事实查询、原理解释、方法比较、数据对比
3. 避免过于简单的是/否问题

请用JSON格式输出，只输出JSON：
{{"question": "问题内容", "ground_truth": "标准答案（2-4句话）"}}"""

        try:
            resp = llm.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=300, temperature=0.7
            )
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r'^```json\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()
            qa = json.loads(raw)
            qa['source_chunk'] = chunk['content'][:300]
            qa['source_doc'] = chunk['doc_id']
            qa['source_type'] = chunk['type']
            qa_pairs.append(qa)
        except Exception:
            continue

    print(f'\u2705 生成测试集: {len(qa_pairs)} 条')
    type_dist = {}
    for qa in qa_pairs:
        t = qa.get('source_type', 'text')
        type_dist[t] = type_dist.get(t, 0) + 1
    print(f'   来源分布: {type_dist}')

    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
        print(f'\U0001f4be 保存至: {save_path}')

    return qa_pairs


def run_ragas_evaluation(
    rag: LiteratureRAG_v4,
    test_questions: List[Dict],
    top_k: int = 5,
    save_path: str = None,
) -> Dict:
    print(f'\U0001f504 开始评估 {len(test_questions)} 条问题...')

    questions, answers, contexts, ground_truths = [], [], [], []

    for i, qa in enumerate(test_questions):
        print(f'  [{i+1}/{len(test_questions)}] {qa["question"][:50]}...')
        result = rag.ask(qa['question'], top_k=top_k, use_multimodal=False)
        questions.append(qa['question'])
        answers.append(result['answer'])
        contexts.append([s['content'] for s in result['sources']])
        ground_truths.append(qa['ground_truth'])

    dataset = Dataset.from_dict({
        'question': questions, 'answer': answers,
        'contexts': contexts, 'ground_truth': ground_truths,
    })

    judge_llm = ChatOpenAI(
        model=LLM_MODEL, api_key=LLM_API_KEY, base_url=LLM_BASE_URL,
    )
    judge_emb = OpenAIEmbeddings(
        model='text-embedding-3-small', api_key=LLM_API_KEY,
    )

    scores = evaluate(
        dataset,
        metrics=[context_recall, context_precision, faithfulness, answer_relevancy],
        llm=judge_llm, embeddings=judge_emb,
    )

    result_dict = scores.to_pandas().mean().to_dict()

    print('\n' + '='*60)
    print('\U0001f4ca RAGAS 评估报告 (v4)')
    print('='*60)
    metrics_info = {
        'context_recall':    ('检索召回率', '检索内容是否覆盖答案所需信息'),
        'context_precision': ('检索精准率', '检索结果中噪声比例'),
        'faithfulness':      ('答案忠实度', '生成答案是否基于检索内容（幻觉检测）'),
        'answer_relevancy':  ('答案相关性', '答案是否真正回答了问题'),
    }
    for key, (cn_name, desc) in metrics_info.items():
        val = result_dict.get(key, 0)
        bar = '\u2588' * int(val * 20) + '\u2591' * (20 - int(val * 20))
        status = '\u2705' if val >= 0.7 else ('\u26a0\ufe0f' if val >= 0.5 else '\u274c')
        print(f'{status} {cn_name:8s} [{bar}] {val:.3f}')
        print(f'   {desc}')

    print('\n\U0001f4a1 问题诊断建议:')
    if result_dict.get('context_recall', 1) < 0.6:
        print('  \u274c 检索召回率低 \u2192 增大 TOP_K_RETRIEVE，或检查chunking粒度')
    if result_dict.get('context_precision', 1) < 0.6:
        print('  \u274c 检索精准率低 \u2192 噪声多，调高Reranker阈值')
    if result_dict.get('faithfulness', 1) < 0.7:
        print('  \u274c 答案忠实度低 \u2192 降低temperature或加强system_prompt')
    if result_dict.get('answer_relevancy', 1) < 0.7:
        print('  \u274c 答案相关性低 \u2192 检查prompt或query理解')

    if save_path:
        scores.to_pandas().to_csv(save_path, index=False, encoding='utf-8')
        print(f'\n\U0001f4be 详细结果: {save_path}')

    return result_dict


def quick_faithfulness_check(rag: LiteratureRAG_v4, queries: List[str]) -> None:
    """快速幻觉检测（无需标注数据）"""
    llm = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)
    print('\U0001f50d 快速幻觉检测')
    print('='*60)

    for query in queries:
        result = rag.ask(query, use_multimodal=False)
        context = '\n'.join(s['content'][:300] for s in result['sources'])

        judge_prompt = f"""判断以下「回答」中的每个陈述是否能在「检索内容」中找到直接支撑。

检索内容：
{context}

回答：
{result['answer']}

请逐句分析，输出JSON：
{{"supported": ["有支撑的句子"], "unsupported": ["无支撑的句子（可能是幻觉）"], "score": 0.0-1.0}}"""

        resp = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role': 'user', 'content': judge_prompt}],
            max_tokens=500, temperature=0
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```json\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()
        try:
            judgment = json.loads(raw)
            score = judgment.get('score', 0)
            status = '\u2705' if score >= 0.8 else ('\u26a0\ufe0f' if score >= 0.6 else '\u274c')
            print(f'{status} [{score:.2f}] {query[:50]}...')
            if judgment.get('unsupported'):
                print(f'  \u26a0\ufe0f  疑似幻觉: {judgment["unsupported"][:2]}')
        except Exception:
            print(f'\u2753 解析失败: {query[:50]}...')


print('\u2705 评估模块就绪')

✅ 评估模块就绪


/tmp/ipykernel_16987/2804365003.py:8: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/tmp/ipykernel_16987/2804365003.py:8: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_16987/2804365003.py:8: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_16987/2804365003.py:8: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be remov

In [ ]:
# ============================================================
# Cell 10: 完整使用流程
# ============================================================

# ============================================================
# 第一次：建索引（只需执行一次，约30-60分钟视文档量）
# ============================================================
all_chunks, doc_trees = load_all_documents_v4(
    MINERU_DIR,
    use_vlm=True,           # 对化学图调VLM
    use_table_summary=True,  # 对表格生成LLM摘要
)
build_index_v4(all_chunks, doc_trees, INDEX_DIR, batch_size=64)

# ============================================================
# 日常使用：加载索引
# ============================================================
rag = LiteratureRAG_v4(index_dir=INDEX_DIR)

# ============================================================
# 基本问答
# ============================================================
result = rag.ask("Which photoinitiators work better under LED light sources?")
print(f'生成模式: {result["gen_mode"]} | 图片: {result["n_images"]} | 表格: {result["n_tables"]}')
print(result['answer'])

📷 多模态处理: VLM=kimi/kimi-k2.5 | 表格摘要=deepseek-chat
📂 发现 904 个文档


解析文档:   1%|          | 8/904 [10:37<19:49:30, 79.65s/it]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 11: 调试 & 检索质量分析
# ============================================================

# 三路召回对比
# rag.debug_retrieve("TPO光引发剂的合成方法有哪些？")

# 测试表格检索
# rag.debug_retrieve("哪种催化剂在Suzuki偶联反应中产率最高？")

# 测试图片检索
# rag.debug_retrieve("TPO的光解机理是什么？")

In [ ]:
# ============================================================
# Cell 12: 评估（可选）
# ============================================================

# ---- 自动生成测试集 ----
# test_qs = generate_test_questions(
#     rag, n_questions=50,
#     save_path=f"{EVAL_DIR}/test_questions_v4.json"
# )

# ---- 运行RAGAS评估 ----
# scores = run_ragas_evaluation(
#     rag, test_qs, top_k=5,
#     save_path=f"{EVAL_DIR}/ragas_results_v4.csv"
# )

# ---- 快速幻觉检测 ----
# quick_faithfulness_check(rag, [
#     "LED光源下效果最好的光引发剂是什么？",
#     "Table 1中最优反应条件是什么？",
#     "Scheme 2展示了哪几种合成路线？",
# ])